# 06 · Q1 : facteurs de performance et rentabilité

**Projet 4 : Analyse des données de vente et retours** · Sprint 4 · US17 et US35

**Question de recherche Q1** : *quels sont les facteurs déterminants de la performance des ventes (boutique, produit, promotion) ?*

**Objectif de l'énoncé servi par ce notebook** : analyser la profitabilité par produit et par boutique, puis chiffrer les leviers de rentabilité.

**Le cadre d'analyse** : une entreprise commande une analyse pour régler sa situation actuelle. Les chiffres de **situation** (résultat, charges, gains possibles) portent sur le dernier exercice, **2021**. Ils sont suivis sur les **trois exercices** : **2019**, l'année de référence où tous les magasins gagnaient de l'argent, **2020**, l'année de bascule, et **2021**. Les **mécanismes** (poids des facteurs, saisonnalité, corrélations) sont mesurés sur les trois années réunies : plus d'observations donnent des mesures plus solides.

**Deux notebooks pour Q1**

| Notebook | Question |
|---|---|
| `06b_q1_chute_des_ventes` | Pourquoi les ventes des magasins ont-elles chuté de moitié ? (nouveautés, descente en gamme, rotation des produits, grands comptes) |
| `06_q1_facteurs_rentabilite` (celui-ci) | Quels facteurs font les ventes et la rentabilité, qui gagne de l'argent, et quels leviers actionner ? |

| Étape | Ce qu'on analyse | Question posée |
|---|---|---|
| 1. Chargement | Ventes avec variables, charges, table commune ventes et charges | |
| 2. Vue d'ensemble | Chiffres clés et décomposition des ventes sur les trois exercices, par canal ; conclusions de `06b` | D'où vient la baisse du chiffre d'affaires ? |
| 3. Le temps | Mois, jour de la semaine, jours fériés | L'activité est-elle saisonnière ? |
| 4. Les produits | Catégorie, gamme, marque, concentration | Quels produits font le chiffre d'affaires et la marge ? |
| 5. Les boutiques | Type, pays, magasins entre eux, surface et effectif | Qu'est-ce qui distingue une boutique d'une autre ? |
| 6. Le poids des facteurs | Part de la variation expliquée (η²), corrélations | Quel facteur compte le plus ? |
| 7. La rentabilité après charges | Résultat par type et par exercice, magasins bénéficiaires, pont du résultat, publicité, mois, catégories | Qui gagne vraiment de l'argent, et qu'est-ce qui a changé ? |
| 8. Les leviers sur l'exercice 2021 | Seuil de rentabilité, publicité, frais généraux, regroupement, remises, retours, gamme, renouvellement de l'assortiment | Que peut-on changer, et pour quel gain ? |
| 9. Synthèse | Réponse à Q1 | |

**Lit dans** : `data/clean/fait_ventes_variables.parquet`, `data/clean/fait_charges.parquet`, `data/clean/fait_stock.parquet`, `data/clean/agregats/ventes_charges_boutique_mois.parquet`, `data/clean/dimensions`

**Écrit dans** : `reports/figures` (figures `06_*.png`) et `reports/livrables/L4_analyse/q1_tableaux.xlsx`

**Utilise** : `src/analyse.py` (fonctions communes aux notebooks d'analyse)

**Ce que ce notebook ne traite pas** : l'effet des promotions (`07_q2_promotions`), les causes des retours (`08_q3_q4_retours`), les niveaux de remise (`09_q5_remises`).

### Vocabulaire utilisé dans tout le notebook

| Terme | Définition |
|---|---|
| Chiffre d'affaires net | montant des ventes moins montant des retours |
| Marge nette des retours | chiffre d'affaires net moins coût des unités vendues et conservées. C'est une marge **avant** frais généraux, publicité et impôts |
| Taux de marge | marge nette des retours ÷ chiffre d'affaires net |
| Frais généraux | administration, informatique, personnel, énergie et communication, loyers, autres charges |
| Résultat d'exploitation | marge nette des retours − frais généraux − publicité |
| Ligne | le cumul d'un produit, dans une boutique, un jour donné, pour une promotion. Ce n'est pas un ticket de caisse |
| Magasin ouvert toute l'année | magasin ouvert au 1er janvier de l'exercice (`05_variables`). Seuls ces magasins sont comparés entre eux |

## 0. Préparation

In [ ]:
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config  -> chemins, années de l'étude (ANNEES_ETUDE, ANNEE_REFERENCE, EXERCICE)
#   viz     -> charte graphique
#   analyse -> resumer (indicateurs par groupe), afficher (tableaux), eta2 (poids d'un facteur),
#              exporter_tableaux, et les libellés français
from src import config, viz, analyse
from src.analyse import LIBELLES_TYPES, ORDRE_TYPES, NOMS_MOIS, resumer, afficher, eta2

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---------------------------------------------------------------------------
# 3. Les trois exercices
# ---------------------------------------------------------------------------
ANNEES = config.ANNEES_ETUDE                  # [2019, 2020, 2021]
REFERENCE, EXERCICE = config.ANNEE_REFERENCE, config.EXERCICE
BASCULE = [a for a in ANNEES if a not in (REFERENCE, EXERCICE)][0]
COULEURS_ANNEES = {REFERENCE: viz.GRIS, BASCULE: viz.CATEGORIES[2], EXERCICE: viz.PRINCIPALE}
print("Racine du projet :", config.ROOT)
print(f"Exercices : {ANNEES} · référence {REFERENCE} · exercice étudié {EXERCICE}")


def m_dollars(valeur, decimales=1, signe=False):
    """Montant en millions de dollars, virgule française : 12,3 M$."""
    texte = f"{valeur / 1e6:{'+' if signe else ''},.{decimales}f}".replace(",", " ").replace(".", ",")
    return f"{texte} M$"


# Tableaux à exporter vers Excel à la fin : nom de l'onglet -> tableau
tableaux = {}

## 1. Chargement

In [ ]:
chemins = {
    "ventes": config.CLEAN / "fait_ventes_variables.parquet",
    "charges": config.CLEAN / "fait_charges.parquet",
    "stock": config.CLEAN / "fait_stock.parquet",
    "commun": config.AGREGATS / "ventes_charges_boutique_mois.parquet",
    "dim_boutique": config.CLEAN_DIMENSIONS / "dim_boutique.parquet",
    "dim_produit": config.CLEAN_DIMENSIONS / "dim_produit.parquet",
}
for nom, chemin in chemins.items():
    if not chemin.exists():
        raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord les notebooks 00 à 05c.")

ventes = pd.read_parquet(chemins["ventes"])
charges = pd.read_parquet(chemins["charges"])
commun = pd.read_parquet(chemins["commun"])
dim_boutique = pd.read_parquet(chemins["dim_boutique"])
dim_produit = pd.read_parquet(chemins["dim_produit"])

# Type de boutique en français, dans les trois tables
ventes["type"] = ventes.type_boutique.map(LIBELLES_TYPES)
commun["type"] = commun.type_boutique.map(LIBELLES_TYPES)
reel = charges[charges.scenario == "Réel"].copy()
reel["type"] = reel.type_boutique.map(LIBELLES_TYPES)

# Magasin ouvert toute l'année : ouvert au 1er janvier de l'année de la vente
ouverture = dim_boutique.set_index("id_boutique").date_ouverture
ventes["annee_complete"] = ventes.id_boutique.map(ouverture) < pd.to_datetime(ventes.annee.astype(str) + "-01-02")

print(f"Ventes  : {len(ventes):,} lignes · {ventes.id_boutique.nunique()} boutiques · {ventes.id_produit.nunique()} produits".replace(",", " "))
print(f"Période : du {ventes.date.min().date()} au {ventes.date.max().date()}")
print(f"Table commune ventes et charges : {len(commun):,} lignes (boutique × mois)".replace(",", " "))

# Garde-fous : trois années complètes, et une table commune qui porte sur les mêmes ventes
if sorted(ventes.annee.unique()) != ANNEES or ventes.groupby("annee").trimestre.nunique().min() != 4:
    raise ValueError(f"La table des ventes ne couvre pas les années {ANNEES} en entier : relancer python -m src.extraction puis les notebooks 00 à 05c.")
if abs(commun.ca_net.sum() - ventes.ca_net.sum()) > 1:
    raise ValueError("La table commune ventes et charges ne correspond pas à la table des ventes : relancer 05b_charges.")

## 2. Vue d'ensemble

On pose les ordres de grandeur, puis on suit les trois exercices.

**Une décomposition utile.** Le chiffre d'affaires est le produit de trois quantités :

> chiffre d'affaires = nombre de lignes × unités par ligne × prix moyen par unité

Quand le chiffre d'affaires change d'un exercice à l'autre, cette décomposition dit si c'est parce qu'on a vendu moins souvent (moins de lignes), par plus petites quantités, ou des produits moins chers.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Chiffres clés, par exercice
# ---------------------------------------------------------------------------
par_exercice = ventes.groupby("annee").agg(ca_brut=("ca_brut", "sum"), retours=("montant_retour", "sum"),
                                           ca_net=("ca_net", "sum"), cout_net=("cout_net", "sum"),
                                           marge_nette=("marge_nette", "sum"), remises=("montant_remise", "sum"),
                                           lignes=("id_vente", "size"), unites=("quantite_vente", "sum"))
par_exercice["taux de marge"] = par_exercice.marge_nette / par_exercice.ca_net
par_exercice["unités par ligne"] = par_exercice.unites / par_exercice.lignes
par_exercice["prix moyen par unité"] = par_exercice.ca_net / par_exercice.unites
chiffres_cles = par_exercice.T
for a, b in zip(ANNEES[:-1], ANNEES[1:]):
    chiffres_cles[f"{a} → {b}"] = chiffres_cles[b] / chiffres_cles[a] - 1
chiffres_cles[f"{REFERENCE} → {EXERCICE}"] = chiffres_cles[EXERCICE] / chiffres_cles[REFERENCE] - 1
chiffres_cles.loc["taux de marge", [c for c in chiffres_cles.columns if "→" in str(c)]] = np.nan
FORMAT_EVOL = {c: "{:+.1%}" for c in chiffres_cles.columns if "→" in str(c)}
print("Chiffres clés par exercice :")
display(chiffres_cles.style.format({a: "{:,.2f}" for a in ANNEES} | FORMAT_EVOL, na_rep=""))
tableaux["2_chiffres_cles"] = chiffres_cles

# Vérification de la décomposition : le produit des trois évolutions redonne celle du chiffre d'affaires
for a, b in [(REFERENCE, BASCULE), (BASCULE, EXERCICE), (REFERENCE, EXERCICE)]:
    e = chiffres_cles[f"{a} → {b}"]
    produit = (1 + e["lignes"]) * (1 + e["unités par ligne"]) * (1 + e["prix moyen par unité"]) - 1
    print(f"{a} → {b} : produit des trois évolutions {produit:+.1%} · évolution du chiffre d'affaires net {e['ca_net']:+.1%}")

# ---------------------------------------------------------------------------
# 2. La même décomposition, par canal
# ---------------------------------------------------------------------------
canal = ventes.groupby(["type", "annee"]).agg(lignes=("id_vente", "size"), unites=("quantite_vente", "sum"),
                                              ca_net=("ca_net", "sum"), marge_nette=("marge_nette", "sum"))
canal["unités par ligne"] = canal.unites / canal.lignes
canal["prix moyen par unité"] = canal.ca_net / canal.unites
canal["taux de marge"] = canal.marge_nette / canal.ca_net
canal = canal.loc[ORDRE_TYPES]
print("Par canal et par exercice :")
display(canal.style.format({"lignes": "{:,.0f}", "unites": "{:,.0f}", "ca_net": "{:,.0f}", "marge_nette": "{:,.0f}",
                            "unités par ligne": "{:.1f}", "prix moyen par unité": "{:,.2f}", "taux de marge": "{:.1%}"}))
evolution_canal = (canal.xs(EXERCICE, level="annee") / canal.xs(REFERENCE, level="annee") - 1)[["ca_net", "lignes", "unités par ligne", "prix moyen par unité"]]
print(f"Évolution {REFERENCE} → {EXERCICE}, par canal :")
display(evolution_canal.style.format("{:+.1%}"))
tableaux["2_canaux"] = canal
tableaux["2_canaux_evolution"] = evolution_canal

# ---------------------------------------------------------------------------
# 3. Graphique : chiffre d'affaires net par canal et par exercice
# ---------------------------------------------------------------------------
ca_canal = canal.ca_net.unstack("annee").loc[ORDRE_TYPES]
fig, ax = plt.subplots(figsize=(9.5, 4.4))
largeur = 0.26
for i, a in enumerate(ANNEES):
    x = np.arange(len(ORDRE_TYPES)) + (i - 1) * largeur
    ax.bar(x, ca_canal[a], width=largeur, color=COULEURS_ANNEES[a], label=str(a))
    for xi, v in zip(x, ca_canal[a]):
        ax.text(xi, v, f"{v / 1e6:,.0f}".replace(",", " "), ha="center", va="bottom", fontsize=9, color=viz.FONCE)
ax.set_xticks(range(len(ORDRE_TYPES)), ORDRE_TYPES)
viz.format_dollars(ax, axe="y")
ax.legend(frameon=False)
viz.titre(ax, "Les magasins perdent la moitié de leur chiffre d'affaires, la boutique en ligne progresse",
          "Chiffre d'affaires net des retours par canal, en millions de dollars")
print("Figure enregistrée :", viz.enregistrer(fig, "06_ca_par_canal"))
plt.show()

**Lecture**

**Le chiffre d'affaires recule sur les deux exercices, la marge suit exactement.**

| | 2019 | 2020 | 2021 | 2019 → 2021 |
|---|---|---|---|---|
| Chiffre d'affaires net | 945,4 M$ | 759,1 M$ | 688,2 M$ | −27,2 % |
| Évolution sur l'exercice | | −19,7 % | −9,3 % | |
| Taux de marge | 56,1 % | 56,1 % | 56,3 % | stable |

Le taux de marge ne bouge pas : la baisse ne vient ni des prix d'achat, ni d'une guerre des prix. On vend moins, à marge égale.

**La décomposition dit comment on vend moins.**

| | 2019 → 2020 | 2020 → 2021 | 2019 → 2021 |
|---|---|---|---|
| Nombre de lignes | −38,7 % | −18,9 % | −50,3 % |
| Unités par ligne | +38,3 % | +44,2 % | +99,4 % |
| Prix moyen par unité | −5,2 % | −22,5 % | −26,5 % |
| Unités vendues | −15,3 % | +17,0 % | −0,9 % |

Les deux exercices ne racontent pas la même chose :

- **2020 est une année de volume** : on vend moins souvent (−38,7 % de lignes) et moins d'unités (−15,3 %), à un prix presque inchangé (−5,2 %) ;
- **2021 est une année de prix** : le volume revient (+17,0 % d'unités, autant qu'en 2019), mais sur des produits beaucoup moins chers (−22,5 % de prix moyen). C'est la descente en gamme étudiée dans `06b`.

**Par canal, deux trajectoires opposées.**

| Canal | 2019 | 2020 | 2021 | 2019 → 2021 | Lignes | Part du CA 2019 → 2021 |
|---|---|---|---|---|---|---|
| Magasins | 511,5 M$ | 306,2 M$ | 263,2 M$ | −48,6 % | −65,5 % | 54,1 % → 38,2 % |
| En ligne | 251,9 M$ | 274,9 M$ | 268,8 M$ | +6,7 % | −21,3 % | 26,6 % → 39,1 % |
| Revendeur | 181,9 M$ | 178,0 M$ | 156,2 M$ | −14,1 % | −30,8 % | 19,2 % → 22,7 % |

En 2021, la boutique en ligne vend plus que les 54 magasins réunis. Les trois canaux ont la même évolution du panier (unités par ligne de +75 % à +103 %, prix moyen de −18 % à −33 %). Ce qui distingue les magasins, c'est la **perte de lignes** : −65,5 %, contre −21,3 % en ligne.

*Interprétation métier (hypothèse)* : le recul des magasins est d'abord une perte d'occasions de vente (moins de passages, ou moins de produits qui trouvent un acheteur en rayon), pas une baisse du panier. `06b` relie cette perte au vieillissement de l'assortiment.

### Pourquoi les ventes ont chuté : ce qu'établit `06b_q1_chute_des_ventes`

La cause de la chute est étudiée en détail dans `06b`. Ses conclusions, qui servent au reste de ce notebook :

| Constat | Verdict |
|---|---|
| La chute des magasins est générale (tous les magasins, tous les mois, toutes les catégories) et porte sur les volumes, surtout sur les produits à plus de 100 $ | Retenue |
| Le catalogue se renouvelle moins : 552 lancements en 2019, 429 en 2020, 272 en 2021, aucun au second semestre 2021. Les nouveautés de l'année portent 69 % de la chute des magasins | Retenue |
| Les clients descendent en gamme, dans tous les canaux et sur les mêmes produits | Retenue |
| Chaque produit se vend dans deux fois moins de magasins ; là où il se vend, il recule comme en ligne | Retenue |
| Un produit non vendu dans un magasin est plus probablement en rayon sans tourner (stock dormant) que retiré de l'assortiment | Indice |
| Les grands comptes de la boutique en ligne partent quand leurs achats de nouveautés s'effondrent | Retenue |

Le fil conducteur est **un assortiment qui vieillit faute de nouveautés**. La recommandation qui en découle, renouveler l'assortiment, est chiffrée en section 8 avec les autres leviers.

## 3. Le temps

Trois questions : y a-t-il une saison forte, des jours forts, et un effet des jours fériés ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Chiffre d'affaires net par mois, les trois exercices superposés
# ---------------------------------------------------------------------------
par_mois = ventes.pivot_table(index="mois", columns="annee", values="ca_net", aggfunc="sum")
fig, ax = plt.subplots(figsize=(10, 4.2))
for a in ANNEES:
    ax.plot(par_mois.index, par_mois[a], color=COULEURS_ANNEES[a], linewidth=2.2, marker="o", label=str(a))
ax.set_xticks(range(1, 13), NOMS_MOIS)
ax.set_ylim(0, par_mois.max().max() * 1.12)
viz.format_dollars(ax, axe="y")
ax.legend(frameon=False, loc="lower right")
viz.titre(ax, "Le même calendrier chaque année, à un niveau plus bas",
          "Chiffre d'affaires net des retours par mois · 56 boutiques")
print("Figure enregistrée :", viz.enregistrer(fig, "06_ca_mensuel"))
plt.show()

# Indice de saisonnalité : chiffre d'affaires du mois ÷ moyenne mensuelle de l'année (100 = mois moyen)
indice = (par_mois / par_mois.mean() * 100).round(0)
indice.index = NOMS_MOIS
print("Indice de saisonnalité (100 = mois moyen de l'année) :")
display(indice.T.style.format("{:.0f}"))
print("Écart moyen entre les profils mensuels des exercices (points d'indice) :",
      {f"{a} / {b}": round(float((indice[a] - indice[b]).abs().mean()), 1) for a, b in [(REFERENCE, BASCULE), (BASCULE, EXERCICE), (REFERENCE, EXERCICE)]})
tableaux["3_mois"] = par_mois.join(indice.set_axis(par_mois.index).add_prefix("indice_"))

# ---------------------------------------------------------------------------
# 2. Jour de la semaine et jours fériés
# ---------------------------------------------------------------------------
par_jour = ventes.groupby("jour_semaine").agg(nom_jour=("nom_jour", "first"), ca_net=("ca_net", "sum"))
par_jour["part_du_ca"] = par_jour.ca_net / par_jour.ca_net.sum()
print("Part du chiffre d'affaires par jour de la semaine (une répartition uniforme donnerait 14,3 %) :")
display(par_jour.set_index("nom_jour").T.loc[["part_du_ca"]].style.format("{:.2%}"))

# Jours fériés : magasins ouverts toute l'année, activité par magasin et par jour
magasins_lignes = ventes[(ventes.type == "Magasins") & ventes.annee_complete]
magasin_jour = (magasins_lignes.groupby(["id_boutique", "date"])
                .agg(ca_net=("ca_net", "sum"), lignes=("id_vente", "size"), ferie=("est_ferie", "max")).reset_index())
feries = magasin_jour.groupby("ferie").agg(jours_magasins=("date", "size"), lignes_par_jour=("lignes", "mean"), ca_par_jour=("ca_net", "mean"))
feries.index = feries.index.map({False: "jour ordinaire", True: "jour férié"})
print("Magasins : activité un jour férié et un jour ordinaire (par magasin et par jour avec au moins une vente) :")
display(feries.style.format({"jours_magasins": "{:,.0f}", "lignes_par_jour": "{:.2f}", "ca_par_jour": "{:,.0f}"}))
r_ferie_ca = magasin_jour.ca_net.corr(magasin_jour.ferie.astype(float))
print(f"Corrélation entre jour férié et chiffre d'affaires du jour (magasin × jour, {len(magasin_jour):,} observations)".replace(",", " ") + f" : {r_ferie_ca:+.3f}")

**Lecture**

**Le même calendrier chaque année, à un niveau plus bas.**

| Indice (100 = mois moyen) | Janv. | Févr. à juill. | Août, sept. | Oct. à déc. |
|---|---|---|---|---|
| 2019 | 69 | 102 à 127 | 97 à 99 | 90 à 93 |
| 2020 | 71 | 110 à 117 | 94 à 98 | 86 à 88 |
| 2021 | 75 | 98 à 118 | 100 à 104 | 89 à 91 |

- **Janvier est le mois le plus faible** des trois exercices (indice 69 à 75) ; le printemps et l'été sont au-dessus de la moyenne.
- **Il n'y a pas de pic de fin d'année** : octobre à décembre restent sous la moyenne les trois années. C'est un point à garder en tête pour la publicité des Fêtes (section 8).
- Les profils mensuels ne s'écartent que de 5,5 à 7,4 points d'indice d'un exercice à l'autre. **La baisse est un changement de niveau, pas un changement de saison** : aucun mois n'explique à lui seul la chute.

**Ni jour fort, ni effet des jours fériés.**

- Chaque jour de la semaine fait entre 14,20 % et 14,36 % du chiffre d'affaires, pour 14,3 % attendus d'une répartition égale.
- Un magasin fait 18 407 $ un jour férié et 19 240 $ un jour ordinaire, avec le même nombre de lignes (6,76 contre 6,79). La corrélation est nulle (−0,011 sur 55 428 jours-magasins).

**Ce qu'on en retient pour la décision** : il n'y a pas de levier calendaire à attendre (horaires, ouverture le dimanche ou les jours fériés, concentration des efforts sur un jour). Les leviers sont dans l'offre et dans les charges.

## 4. Les produits

Quatre angles : la catégorie, la gamme (économique, standard, luxe), la marque, et la concentration (quelques produits font-ils l'essentiel du chiffre d'affaires ?).

In [ ]:
# ---------------------------------------------------------------------------
# 1. Par catégorie, sur les trois exercices
# ---------------------------------------------------------------------------
par_categorie = resumer(ventes, "categorie").sort_values("ca_net", ascending=False)
afficher(par_categorie)
tableaux["4_categories"] = par_categorie

cat_annee = ventes.pivot_table(index="categorie", columns="annee", values=["ca_net", "marge_nette"], aggfunc="sum")
evol_cat = pd.DataFrame({f"ca_net_{a}": cat_annee[("ca_net", a)] for a in ANNEES})
evol_cat[f"évolution {REFERENCE} → {EXERCICE}"] = evol_cat[f"ca_net_{EXERCICE}"] / evol_cat[f"ca_net_{REFERENCE}"] - 1
evol_cat[f"évolution {BASCULE} → {EXERCICE}"] = evol_cat[f"ca_net_{EXERCICE}"] / evol_cat[f"ca_net_{BASCULE}"] - 1
for a in ANNEES:
    evol_cat[f"taux de marge {a}"] = cat_annee[("marge_nette", a)] / cat_annee[("ca_net", a)]
evol_cat["part de la baisse " + f"{REFERENCE} → {EXERCICE}"] = ((evol_cat[f"ca_net_{EXERCICE}"] - evol_cat[f"ca_net_{REFERENCE}"])
                                                               / (evol_cat[f"ca_net_{EXERCICE}"] - evol_cat[f"ca_net_{REFERENCE}"]).sum())
evol_cat = evol_cat.sort_values(f"ca_net_{REFERENCE}", ascending=False)
print("Par catégorie et par exercice :")
display(evol_cat.style.format({c: ("{:,.0f}" if c.startswith("ca_net") else "{:+.1%}" if "évolution" in c else "{:.1%}") for c in evol_cat.columns}))
tableaux["4_categories_par_exercice"] = evol_cat

# ---------------------------------------------------------------------------
# 2. Graphique : chiffre d'affaires et taux de marge par catégorie, exercice et référence
# ---------------------------------------------------------------------------
trie = evol_cat.sort_values(f"ca_net_{EXERCICE}")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.6), sharey=True, gridspec_kw={"width_ratios": [3, 2]})
y = np.arange(len(trie))
ax1.barh(y + 0.2, trie[f"ca_net_{REFERENCE}"], height=0.38, color=COULEURS_ANNEES[REFERENCE], label=str(REFERENCE))
ax1.barh(y - 0.2, trie[f"ca_net_{EXERCICE}"], height=0.38, color=COULEURS_ANNEES[EXERCICE], label=str(EXERCICE))
ax1.set_yticks(y, trie.index)
viz.format_dollars(ax1, axe="x")
ax1.legend(frameon=False, loc="lower right")
viz.titre(ax1, "Toutes les grandes catégories reculent", f"Chiffre d'affaires net des retours, {REFERENCE} et {EXERCICE}")
ax2.barh(y + 0.2, trie[f"taux de marge {REFERENCE}"], height=0.38, color=COULEURS_ANNEES[REFERENCE])
ax2.barh(y - 0.2, trie[f"taux de marge {EXERCICE}"], height=0.38, color=COULEURS_ANNEES[EXERCICE])
ax2.set_xlim(0.45, 0.65)
viz.format_pourcent(ax2, axe="x")
viz.titre(ax2, "Le taux de marge ne bouge pas", "Taux de marge nette des retours")
print("Figure enregistrée :", viz.enregistrer(fig, "06_categories"))
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 3. Par gamme (classe) et par marque
# ---------------------------------------------------------------------------
LIBELLES_CLASSES = {"Economy": "Économique", "Regular": "Standard", "Deluxe": "Luxe"}
ventes["gamme"] = ventes.classe.map(LIBELLES_CLASSES)
par_classe = resumer(ventes, "gamme").loc[["Économique", "Standard", "Luxe"]]
print("Par gamme, trois exercices réunis :")
afficher(par_classe)
gamme_annee = ventes.pivot_table(index="gamme", columns="annee", values="ca_net", aggfunc="sum").loc[["Économique", "Standard", "Luxe"]]
print("Part de chaque gamme dans le chiffre d'affaires, par exercice :")
display((gamme_annee / gamme_annee.sum()).style.format("{:.1%}"))
tableaux["4_gammes"] = par_classe.join(gamme_annee.add_prefix("ca_net_"))

par_marque = resumer(ventes, "marque").sort_values("ca_net", ascending=False)
marque_annee = ventes.pivot_table(index="marque", columns="annee", values="ca_net", aggfunc="sum")
par_marque[f"évolution {REFERENCE} → {EXERCICE}"] = marque_annee[EXERCICE] / marque_annee[REFERENCE] - 1
print("Par marque :")
afficher(par_marque, {f"évolution {REFERENCE} → {EXERCICE}": "{:+.1%}"})
tableaux["4_marques"] = par_marque

# ---------------------------------------------------------------------------
# 4. Concentration : courbe de Pareto des produits, sur l'exercice
# ---------------------------------------------------------------------------
ventes_ex = ventes[ventes.annee == EXERCICE]
par_produit = (resumer(ventes_ex, ["id_produit", "nom_produit", "categorie"])
               .sort_values("ca_net", ascending=False).reset_index())
par_produit["rang"] = np.arange(1, len(par_produit) + 1)
par_produit["part_des_produits"] = par_produit.rang / len(par_produit)
par_produit["ca_cumule"] = par_produit.ca_net.cumsum() / par_produit.ca_net.sum()
seuils = {s: int((par_produit.ca_cumule < s).sum() + 1) for s in (0.5, 0.8)}
for s, n in seuils.items():
    print(f"{EXERCICE} : {s * 100:.0f} % du chiffre d'affaires est fait par {n} produits, soit {n / len(par_produit) * 100:.0f} % des {len(par_produit)} produits vendus")
# Même mesure sur l'année de référence, pour comparaison
ref_produit = ventes[ventes.annee == REFERENCE].groupby("id_produit").ca_net.sum().sort_values(ascending=False)
ref_cumul = ref_produit.cumsum() / ref_produit.sum()
print(f"{REFERENCE} : 80 % du chiffre d'affaires est fait par {int((ref_cumul < 0.8).sum() + 1)} produits, soit "
      f"{(ref_cumul < 0.8).mean() * 100:.0f} % des {len(ref_produit)} produits vendus")

fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.plot(par_produit.part_des_produits, par_produit.ca_cumule, color=viz.PRINCIPALE, linewidth=2.4)
ax.plot([0, 1], [0, 1], color=viz.GRIS, linestyle=":", linewidth=1.2)
part_80 = seuils[0.8] / len(par_produit)
ax.plot([part_80, part_80], [0, 0.8], color=viz.ALERTE, linestyle="--", linewidth=1.5)
ax.plot([0, part_80], [0.8, 0.8], color=viz.ALERTE, linestyle="--", linewidth=1.5)
ax.text(part_80 + 0.02, 0.4, f"{part_80:.0%} des produits\nfont 80 % des ventes".replace(".", ","), color=viz.ALERTE, va="center")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("Part des produits, du plus vendu au moins vendu")
ax.set_ylabel("Part cumulée du chiffre d'affaires")
viz.format_pourcent(ax, axe="x"); viz.format_pourcent(ax, axe="y")
viz.titre(ax, "Une concentration modérée : pas de règle des 80/20",
          f"Courbe de Pareto des produits, exercice {EXERCICE} · pointillé gris : répartition parfaitement égale")
print("Figure enregistrée :", viz.enregistrer(fig, "06_pareto_produits"))
plt.show()

print(f"Les dix produits au chiffre d'affaires le plus élevé en {EXERCICE} :")
afficher(par_produit.head(10)[["nom_produit", "categorie", "nb_lignes", "quantite", "ca_net", "marge_nette", "taux_de_marge", "part_du_ca"]])
tableaux["4_top_produits"] = par_produit.head(50)

**Lecture**

**Trois catégories font l'essentiel du chiffre d'affaires.** Sur les trois exercices, l'électroménager (31,7 %), les ordinateurs (26,2 %) et les appareils photo (20,5 %) font 78,4 % des ventes. Les cinq autres catégories pèsent chacune moins de 11 %.

**La baisse vient surtout des appareils photo et de l'électroménager.**

| Catégorie | 2019 | 2021 | 2019 → 2021 | 2020 → 2021 | Part de la baisse |
|---|---|---|---|---|---|
| Appareils photo | 228,1 M$ | 112,0 M$ | −50,9 % | −25,5 % | 45,1 % |
| Électroménager | 287,3 M$ | 204,0 M$ | −29,0 % | −23,8 % | 32,4 % |
| Téléphones | 75,5 M$ | 48,0 M$ | −36,4 % | +3,4 % | 10,7 % |
| Ordinateurs | 235,6 M$ | 210,1 M$ | −10,8 % | +15,5 % | 9,9 % |

- Les appareils photo, la catégorie au **taux de marge le plus élevé** des grandes catégories (59,8 % en 2019), font à eux seuls 45 % de la baisse.
- Les ordinateurs repartent en 2021 (+15,5 %) et deviennent la **première catégorie** (210,1 M$ contre 204,0 M$ pour l'électroménager).
- Le taux de marge de chaque catégorie bouge de moins de trois points en deux exercices : la catégorie fixe le niveau de marge, pas son évolution.

**La gamme fixe la marge, et la clientèle descend en gamme.**

| Gamme | Taux de marge | Prix moyen par unité | Part du CA 2019 | 2020 | 2021 |
|---|---|---|---|---|---|
| Économique | 48,9 % | 70 $ | 10,6 % | 14,2 % | 21,6 % |
| Standard | 54,0 % | 344 $ | 64,0 % | 61,7 % | 52,0 % |
| Luxe | 65,5 % | 596 $ | 25,4 % | 24,1 % | 26,4 % |

Le glissement se fait du **standard vers l'économique** ; le luxe garde sa part. Le taux de marge global reste donc stable, mais le montant par unité vendue s'effondre : il faut cinq produits économiques pour faire le chiffre d'affaires d'un produit standard.

**La marque ne fait pas la marge.** Les taux de marge des onze marques vont de 53,0 % à 58,7 %. Les évolutions sont très dispersées (Northwind Traders −70,5 %, A. Datum −51,3 %, Adventure Works +17,0 %, Tailspin Toys +73,2 %) : elles suivent les catégories dans lesquelles chaque marque est présente.

**Pas de produit vedette.** En 2021, 80 % du chiffre d'affaires est fait par 44 % des produits (1 093 sur 2 504), et le premier produit ne pèse que 0,4 % des ventes. En 2019, 80 % était fait par 33 % des produits. La performance ne repose pas sur quelques références : elle tient à la **largeur et au renouvellement de l'assortiment**, ce qui rejoint `06b`. Les dix premiers produits de 2021 sont des ordinateurs portables et deux réfrigérateurs, tous à 53 à 66 % de marge.

## 5. Les boutiques

Les 56 boutiques sont de trois types très différents en taille : 54 magasins, une boutique en ligne et un revendeur. On les compare d'abord par type, puis on regarde les magasins entre eux, **sur les seuls magasins ouverts toute l'année**.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Par type de boutique et par exercice
# ---------------------------------------------------------------------------
par_type = resumer(ventes, ["type", "annee"]).loc[ORDRE_TYPES]
par_type.insert(0, "nb_boutiques", ventes.groupby(["type", "annee"]).id_boutique.nunique())
par_type["ca_par_boutique"] = par_type.ca_net / par_type.nb_boutiques
par_type["part_du_ca"] = par_type.ca_net / par_type.groupby(level="annee").ca_net.transform("sum")
afficher(par_type, {"nb_boutiques": "{:.0f}", "ca_par_boutique": "{:,.0f}"})
tableaux["5_types"] = par_type

# Le mix de catégories est-il le même dans les trois types ? (exercice)
mix = ventes_ex.pivot_table(index="categorie", columns="type", values="ca_net", aggfunc="sum")[ORDRE_TYPES]
mix = mix / mix.sum()
print(f"Part de chaque catégorie dans le chiffre d'affaires de chaque type de boutique, {EXERCICE} :")
display(mix.sort_values("Magasins", ascending=False).style.format("{:.1%}"))

# ---------------------------------------------------------------------------
# 2. Les magasins entre eux, par exercice (magasins ouverts toute l'année)
# ---------------------------------------------------------------------------
mag_an = resumer(magasins_lignes, ["annee", "id_boutique", "nom_boutique", "pays"]).reset_index()
mag_an = mag_an.merge(dim_boutique[["id_boutique", "surface", "effectif"]], on="id_boutique", how="left")
print("Chiffre d'affaires d'un magasin ouvert toute l'année : dispersion par exercice")
display(mag_an.groupby("annee").ca_net.describe(percentiles=[0.25, 0.5, 0.75])[["count", "min", "25%", "50%", "75%", "max"]]
        .style.format({"count": "{:.0f}"} | {c: "{:,.0f}" for c in ["min", "25%", "50%", "75%", "max"]}))
mag_large = mag_an.pivot_table(index=["nom_boutique", "pays"], columns="annee", values="ca_net")
mag_large[f"évolution {REFERENCE} → {EXERCICE}"] = mag_large[EXERCICE] / mag_large[REFERENCE] - 1
mag_large = mag_large.sort_values(EXERCICE, ascending=False)
FORMAT_MAG = {a: "{:,.0f}" for a in ANNEES} | {f"évolution {REFERENCE} → {EXERCICE}": "{:+.1%}"}
print(f"Les cinq magasins au chiffre d'affaires le plus élevé en {EXERCICE} :")
display(mag_large.head(5).style.format(FORMAT_MAG, na_rep="(non ouvert)"))
print(f"Les cinq magasins au chiffre d'affaires le plus faible en {EXERCICE} :")
display(mag_large.dropna(subset=[EXERCICE]).tail(5).style.format(FORMAT_MAG, na_rep="(non ouvert)"))
tableaux["5_magasins"] = mag_large

# ---------------------------------------------------------------------------
# 3. Par pays, sur l'exercice (magasins seulement : la boutique en ligne et le revendeur servent toute l'Europe)
# ---------------------------------------------------------------------------
mag_ex = mag_an[mag_an.annee == EXERCICE]
par_pays = (mag_ex.groupby("pays").agg(nb_magasins=("id_boutique", "size"), ca_net=("ca_net", "sum"), marge_nette=("marge_nette", "sum"))
            .sort_values("ca_net", ascending=False))
par_pays["part_du_ca"] = par_pays.ca_net / par_pays.ca_net.sum()
par_pays["ca_par_magasin"] = par_pays.ca_net / par_pays.nb_magasins
par_pays["taux_de_marge"] = par_pays.marge_nette / par_pays.ca_net
afficher(par_pays, {"nb_magasins": "{:.0f}", "ca_par_magasin": "{:,.0f}"})
tableaux["5_pays"] = par_pays

# ---------------------------------------------------------------------------
# 4. La surface et l'effectif expliquent-ils le chiffre d'affaires d'un magasin ?
# ---------------------------------------------------------------------------
correlations_moyens = mag_ex[["ca_net", "nb_lignes", "ca_par_ligne", "surface", "effectif"]].corr(method="spearman").loc[
    ["surface", "effectif"], ["ca_net", "nb_lignes", "ca_par_ligne"]]
print(f"Corrélation de Spearman entre les moyens du magasin et son activité, {EXERCICE} ({len(mag_ex)} magasins) :")
display(correlations_moyens.style.format("{:+.2f}"))

**Lecture**

**Trois canaux de taille très différente.** Le chiffre d'affaires d'un magasin passe de 9,47 M$ à 5,78 M$ puis 4,87 M$. La boutique en ligne (268,8 M$ en 2021) vend autant que 55 magasins, le revendeur (156,2 M$) autant que 32.

**Le mix de produits diffère peu.** En 2021, les magasins vendent un peu plus d'ordinateurs (37,6 % de leur chiffre d'affaires, contre 25,2 % en ligne) et moins d'électroménager (26,1 % contre 31,8 %). Les jeux et jouets y sont presque absents (0,2 % contre 3,9 %). Ces écarts n'expliquent pas l'évolution opposée des canaux : `06b` montre que la chute des magasins touche toutes les catégories.

**Les magasins se ressemblent beaucoup.**

| Magasins ouverts toute l'année | 2019 | 2020 | 2021 |
|---|---|---|---|
| Nombre | 48 | 51 | 53 |
| Chiffre d'affaires médian | 10,47 M$ | 5,72 M$ | 4,84 M$ |
| Moitié centrale (25 % à 75 %) | 10,34 à 10,57 M$ | 5,64 à 5,84 M$ | 4,74 à 4,93 M$ |

La moitié des magasins se tient à moins de 2 % de la médiane, chaque exercice. Deux exceptions :

- **Contoso Moscow No.2** : 10,59 → 13,27 → 13,08 M$ (+23,6 %), le seul magasin qui progresse ;
- **Contoso Cambridge** : 9,25 → 7,09 → 5,57 M$ (−39,8 %), deuxième en 2021 parce qu'il a moins reculé que les autres (−50 % à −52 %).

Les magasins les plus faibles en 2021 sont ceux ouverts en 2020 (Glasgow, Düsseldorf, Rome : 4,0 à 4,2 M$ ; Berlin et Marseille : 3,6 à 3,7 M$).

**Le pays ne fait pas la différence.** Le Royaume-Uni (15 magasins, 27,6 % des ventes des magasins), l'Allemagne (11, 20,0 %) et la France (7, 12,5 %) pèsent par leur nombre de magasins. Par magasin, tous les pays sont entre 4,6 et 5,1 M$, sauf la Russie (6,87 M$), tirée par Moscow No.2.

**La surface et l'effectif n'expliquent pas les ventes** (Spearman −0,14 et +0,01 avec le chiffre d'affaires de 2021).

*Interprétation métier (hypothèse)* : les moyens d'un magasin (surface, personnel) ne sont pas dimensionnés sur son activité. On retrouve la même logique pour les frais généraux en section 8 : la charge d'un magasin est reconduite d'un exercice à l'autre, elle ne s'ajuste pas à ce qu'il vend.

## 6. Le poids de chaque facteur

Pour classer les facteurs, on mesure pour chacun la **part de la variation** du chiffre d'affaires qu'il explique à lui seul. C'est le rapport de corrélation (noté η², « êta carré »), la mesure d'effet de l'analyse de variance :

> η² = variation entre les groupes ÷ variation totale

η² vaut 0 si les groupes ont tous la même moyenne (le facteur n'explique rien) et 1 si le groupe suffit à connaître la valeur.

**Pourquoi η² plutôt qu'un test de significativité ?** Avec 715 291 lignes, n'importe quel écart, même infime, est « statistiquement significatif ». La valeur p ne dit donc rien d'utile ici. Ce qui compte est la **taille** de l'effet.

On le calcule à deux niveaux, parce que la réponse n'est pas la même :

- **au niveau de la ligne** : qu'est-ce qui fait qu'une ligne rapporte plus qu'une autre ? (sur le logarithme du chiffre d'affaires, parce que les montants s'étalent de quelques dollars à plus de cent mille) ;
- **au niveau du magasin et du mois** : qu'est-ce qui fait qu'un magasin vend plus qu'un autre un mois donné ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Au niveau de la ligne : logarithme du chiffre d'affaires net de la ligne
# ---------------------------------------------------------------------------
log_ca = np.log(ventes.ca_net.clip(lower=1))     # clip : quelques lignes ont un CA net nul (tout a été retourné)
FACTEURS_LIGNE = {
    "Produit": "id_produit", "Sous-catégorie": "sous_categorie", "Marque": "marque", "Gamme": "classe",
    "Catégorie": "categorie", "Promotion (laquelle)": "id_promotion", "Année": "annee", "Mois": "mois",
    "Boutique": "id_boutique", "Type de boutique": "type", "Promotion (oui ou non)": "a_promotion",
    "Pays": "pays", "Jour de la semaine": "nom_jour", "Jour férié": "est_ferie",
}
poids_ligne = pd.Series({nom: eta2(log_ca, ventes[col]) for nom, col in FACTEURS_LIGNE.items()}).sort_values(ascending=False)
r2_prix = np.corrcoef(np.log(ventes.prix_unitaire), log_ca)[0, 1] ** 2
r2_quantite = np.corrcoef(np.log(ventes.quantite_vente), log_ca)[0, 1] ** 2
print(f"Part de la variation du chiffre d'affaires d'une ligne expliquée par son prix unitaire : {r2_prix:.1%}")
print(f"Part expliquée par la quantité vendue : {r2_quantite:.1%}")

# ---------------------------------------------------------------------------
# 2. Au niveau du magasin et du mois (magasins ouverts toute l'année, mois après ouverture)
# ---------------------------------------------------------------------------
magasins_mois = commun[(commun.type == "Magasins") & commun.annee_complete].copy()
magasins_mois["ca_par_ligne"] = magasins_mois.ca_net / magasins_mois.nb_lignes
FACTEURS_MOIS = {"Année": "annee", "Mois de l'année": "mois", "Boutique": "id_boutique", "Pays": "pays"}
sans_moscou = magasins_mois[~magasins_mois.nom_boutique.str.contains("Moscow No.2")]
poids_mois = pd.DataFrame({
    "tous les magasins": {nom: eta2(magasins_mois.ca_net, magasins_mois[col]) for nom, col in FACTEURS_MOIS.items()},
    "sans Moscow No.2": {nom: eta2(sans_moscou.ca_net, sans_moscou[col]) for nom, col in FACTEURS_MOIS.items()},
}).sort_values("sans Moscow No.2", ascending=False)
r2_lignes = magasins_mois.ca_net.corr(magasins_mois.nb_lignes) ** 2
r2_ca_ligne = magasins_mois.ca_net.corr(magasins_mois.ca_par_ligne) ** 2
print(f"\nMagasins, par mois : part du chiffre d'affaires expliquée par le nombre de lignes : {r2_lignes:.1%}",
      f"· par le chiffre d'affaires par ligne : {r2_ca_ligne:.1%}")

display(poids_ligne.to_frame("niveau ligne (η²)").style.format("{:.1%}"))
print("Niveau magasin et mois (η²) :")
display(poids_mois.style.format("{:.1%}"))
tableaux["6_poids_facteurs_ligne"] = poids_ligne.to_frame("eta2")
tableaux["6_poids_facteurs_mois"] = poids_mois

# ---------------------------------------------------------------------------
# 3. Graphique : poids des facteurs au niveau de la ligne
# ---------------------------------------------------------------------------
trie = poids_ligne.sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(trie.index, trie.values, color=[viz.PRINCIPALE if v >= 0.05 else viz.GRIS for v in trie.values])
for y, v in enumerate(trie.values):
    ax.text(v + 0.01, y, f"{v:.1%}".replace(".", ","), va="center", color=viz.FONCE, fontsize=9)
ax.set_xlim(0, 1)
viz.format_pourcent(ax, axe="x")
viz.titre(ax, "Le montant d'une ligne dépend du produit, presque pas de la boutique ni du jour",
          "Part de la variation du chiffre d'affaires d'une ligne expliquée par chaque facteur (η²) · trois exercices")
print("Figure enregistrée :", viz.enregistrer(fig, "06_poids_facteurs"))
plt.show()

**Lecture**

**Au niveau de la ligne, c'est le produit qui fait le montant.**

| Facteur | Part de la variation expliquée |
|---|---|
| Produit | 86,1 % |
| Prix unitaire (à lui seul) | 77,6 % |
| Sous-catégorie | 58,6 % |
| Gamme | 28,7 % |
| Marque · catégorie | 18,1 % · 13,5 % |
| Quantité vendue | 10,5 % |
| Année · promotion (laquelle) · mois | 5,7 % · 5,5 % · 1,4 % |
| Boutique · type de boutique · pays | 0,6 % · 0,5 % · 0,3 % |
| Promotion (oui ou non) · jour férié · jour de la semaine | 0,2 % · 0,0 % · 0,0 % |

Ce que rapporte une vente dépend presque entièrement de **ce qu'on vend**. Où et quand on le vend n'y change presque rien. La promotion « laquelle » (5,5 %) pèse plus que la promotion « oui ou non » (0,2 %) parce que chaque promotion couvre une période : elle porte en partie l'effet de l'année. Son effet propre est étudié dans `07_q2_promotions`.

**Au niveau du magasin et du mois, c'est l'exercice et le nombre de ventes.**

| Facteur | Tous les magasins | Sans Moscow No.2 |
|---|---|---|
| Année | 69,7 % | 79,0 % |
| Boutique | 10,9 % | 4,3 % |
| Mois de l'année | 9,2 % | 10,5 % |
| Pays | 2,0 % | 0,2 % |

Le nombre de lignes explique à lui seul 76,2 % du chiffre d'affaires d'un magasin un mois donné ; le chiffre d'affaires par ligne, 5,5 %. Sans Moscow No.2, le magasin n'explique plus que 4,3 %.

**Réponse à Q1 sur le poids des facteurs** :

1. au niveau d'une vente, le **produit et son prix** ;
2. au niveau d'un magasin, le **nombre de ventes**, et donc le nombre de produits qui trouvent un acheteur ;
3. d'un exercice à l'autre, **ce qui change pour toute l'entreprise en même temps**. `06b` établit que c'est l'assortiment (moins de nouveautés, moins de produits vendus par magasin).

La boutique, le pays, le jour et la promotion en tant que telle ne sont pas des facteurs de performance.

### Corrélations entre les variables numériques

η² convient aux facteurs qui sont des **catégories**. Pour les variables qui sont des **nombres** (taux de remise, taux de retour, frais généraux, surface), l'outil est le **coefficient de corrélation**, compris entre −1 et +1. On le calcule au niveau **magasin et mois**, avec deux coefficients :

- **Pearson** mesure un lien linéaire. Il est sensible aux valeurs très élevées, donc à *Moscow No.2* ;
- **Spearman** travaille sur les rangs, sans être tiré par un seul magasin.

**Repères de lecture** : en dessous de 0,1 en valeur absolue, pas de lien ; de 0,1 à 0,3, lien faible ; de 0,3 à 0,5, lien modéré ; au-dessus de 0,5, lien fort. Une corrélation ne dit rien du sens de la cause.

La publicité n'est pas dans cette matrice : chaque magasin reçoit le même montant un mois donné, une corrélation entre magasins n'aurait pas de sens. Elle est étudiée à part en section 7. La sévérité du COVID-19 est mesurée sur 2020 et 2021 seulement (elle vaut 0 en 2019).

In [ ]:
# ---------------------------------------------------------------------------
# 1. Table magasin × mois avec les variables numériques
# ---------------------------------------------------------------------------
mm = magasins_mois.copy()
mm["prix_moyen_par_unite"] = mm.ca_net / mm.quantite_vente
mm["taux_de_marge"] = mm.marge_nette / mm.ca_net
mm["taux_de_remise"] = mm.montant_remise / mm.montant_avant_remise      # taux effectif, jamais le taux annoncé
mm["taux_de_retour"] = mm.montant_retour / mm.ca_brut
mm["taux_frais_generaux"] = mm.frais_generaux / mm.ca_net
cle_mois = ["id_boutique", "annee", "mois"]
depuis_lignes = (magasins_lignes.groupby(cle_mois)
                 .agg(part_lignes_en_promotion=("a_promotion", "mean"), severite_covid=("severite_covid", "mean"),
                      part_lignes_jour_ferie=("est_ferie", "mean")).reset_index())
mm = mm.merge(depuis_lignes, on=cle_mois, how="left", validate="one_to_one")
mm = mm.merge(dim_boutique[["id_boutique", "surface", "effectif"]], on="id_boutique", how="left", validate="many_to_one")

VARIABLES = {
    "ca_net": "Chiffre d'affaires net", "nb_lignes": "Nombre de lignes", "ca_par_ligne": "CA par ligne",
    "prix_moyen_par_unite": "Prix moyen par unité", "taux_de_marge": "Taux de marge",
    "taux_de_remise": "Taux de remise effectif", "part_lignes_en_promotion": "Part des lignes en promotion",
    "taux_de_retour": "Taux de retour", "frais_generaux": "Frais généraux", "taux_frais_generaux": "Taux de frais généraux",
    "resultat_exploitation": "Résultat d'exploitation", "part_lignes_jour_ferie": "Part des lignes un jour férié",
    "surface": "Surface", "effectif": "Effectif",
}
donnees = mm[list(VARIABLES)].rename(columns=VARIABLES)
print(f"Table magasin × mois : {len(donnees):,} lignes · {donnees.shape[1]} variables".replace(",", " "))
pearson = donnees.corr(method="pearson")
spearman = donnees.corr(method="spearman")
tableaux["6_correlations_pearson"] = pearson
tableaux["6_correlations_spearman"] = spearman

cibles = pd.DataFrame({
    "CA net · Pearson": pearson["Chiffre d'affaires net"], "CA net · Spearman": spearman["Chiffre d'affaires net"],
    "Résultat d'exploitation · Pearson": pearson["Résultat d'exploitation"],
    "Résultat d'exploitation · Spearman": spearman["Résultat d'exploitation"],
}).drop(index=["Chiffre d'affaires net"])
print("Corrélation de chaque variable avec le chiffre d'affaires net et avec le résultat d'exploitation :")
display(cibles.style.format("{:+.2f}").background_gradient(cmap="RdBu", vmin=-1, vmax=1))
tableaux["6_correlations_cibles"] = cibles

# COVID-19 : sur 2020 et 2021, à mois égal (on retire la moyenne de chaque mois pour ne pas confondre avec la saison)
covid = mm[mm.annee.isin(config.ANNEES_ATTENDUES)].copy()
for col in ["ca_net", "severite_covid"]:
    covid[col + "_centre"] = covid[col] - covid.groupby(["annee", "mois"])[col].transform("mean")
r_covid = covid.ca_net_centre.corr(covid.severite_covid_centre, method="spearman")
print(f"Sévérité COVID-19 et chiffre d'affaires d'un magasin, à mois égal, {config.ANNEES_ATTENDUES} (Spearman) : {r_covid:+.3f}")

# ---------------------------------------------------------------------------
# 2. Carte de chaleur (Spearman)
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10.5, 8.5))
n = len(spearman)
image = ax.pcolormesh(spearman.values, cmap="RdBu", vmin=-1, vmax=1, edgecolors="white", linewidth=0.5)
ax.invert_yaxis()
fig.colorbar(image, ax=ax, fraction=0.04, pad=0.02)
centres = np.arange(n) + 0.5
ax.set_xticks(centres, spearman.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(centres, spearman.index, fontsize=9)
ax.grid(False)
for i in range(n):
    for j in range(n):
        valeur = spearman.values[i, j]
        ax.text(j + 0.5, i + 0.5, f"{valeur:.2f}".replace(".", ","), transform=ax.transData, ha="center", va="center",
                fontsize=7, family="DejaVu Sans", color="white" if abs(valeur) > 0.6 else viz.FONCE)
viz.titre(ax, "Corrélations entre les variables d'un magasin, mois par mois",
          "Coefficient de Spearman · magasins ouverts toute l'année · bleu : lien positif, rouge : lien négatif")
ax.texts[-1].set_y(1.008)
print("Figure enregistrée :", viz.enregistrer(fig, "06_correlations"))
plt.show()

**Lecture**

**Ce qui va avec les ventes et le résultat d'un magasin, mois par mois.**

| Variable | CA net (Pearson · Spearman) | Résultat (Pearson · Spearman) | Lecture |
|---|---|---|---|
| Nombre de lignes | +0,87 · +0,79 | +0,57 · +0,60 | lien fort : vendre plus souvent, c'est vendre plus et gagner plus |
| Taux de frais généraux | −0,79 · −0,81 | −0,52 · −0,55 | lien fort : quand les ventes baissent, le poids des frais généraux monte |
| Prix moyen par unité | +0,36 · +0,42 | +0,33 · +0,37 | lien modéré : les mois où l'on vend plus cher sont plus rentables |
| Taux de marge | +0,25 · +0,34 | +0,34 · +0,35 | lien faible à modéré, même sens |
| Part des lignes en promotion | −0,13 · −0,21 | −0,38 · −0,44 | lien modéré avec le résultat : plus de promotions, moins de résultat |
| Taux de remise effectif | −0,25 · −0,29 | −0,12 · −0,19 | lien faible, même sens |
| Frais généraux (montant) | +0,25 · +0,22 | +0,08 · −0,05 | le montant ne suit presque pas les ventes |
| Taux de retour, surface, effectif, jours fériés | −0,03 à +0,14 | −0,03 à +0,14 | pas de lien |

Trois enseignements :

- **Le mécanisme de la perte est là** : le montant des frais généraux ne suit pas les ventes (+0,25), donc leur poids explose quand les ventes baissent (−0,79). C'est ce qui transforme une baisse de ventes en déficit (section 7).
- **Vendre plus cher rapporte plus** : la descente en gamme (section 4) pèse sur le résultat, pas seulement sur le chiffre d'affaires.
- **Les promotions vont avec moins de résultat**, mais ce n'est pas la promotion qui le fait baisser : `07_q2_promotions` montre qu'à mois égal la corrélation tombe à −0,02. Les promotions tombent pendant les mois où le résultat est bas (mois de publicité, mois faibles de l'année). Leur effet sur les ventes est étudié dans `07_q2_promotions` et `09_q5_remises`.

**La pandémie ne distingue pas un magasin d'un autre.** À mois égal, sur 2020 et 2021, la sévérité locale du COVID-19 n'a aucun lien avec les ventes d'un magasin (Spearman −0,031). La crise a pu peser sur tous les magasins en même temps (consommation des ménages en baisse d'environ 5 % en 2020, `06b`), mais elle n'explique pas pourquoi un magasin vend plus qu'un autre.

## 7. La rentabilité après charges

Grâce au plan stratégique (`05b_charges`), on retire les **frais généraux** et la **publicité** de chaque boutique. La question change : non plus « qui vend le plus », mais « **qui gagne vraiment de l'argent** », et comment cela a évolué sur les trois exercices.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Le compte de résultat par type de boutique et par exercice
# ---------------------------------------------------------------------------
POSTES = ["ca_net", "marge_nette", "frais_generaux", "publicite", "resultat_exploitation", "impots", "resultat_net"]
situation = commun.groupby(["type", "annee"])[POSTES].sum().loc[ORDRE_TYPES]
ensemble = commun.groupby("annee")[POSTES].sum()
ensemble.index = pd.MultiIndex.from_product([["Ensemble"], ensemble.index], names=["type", "annee"])
situation = pd.concat([situation, ensemble])
print("Compte de résultat par type de boutique et par exercice (millions de dollars) :")
display((situation / 1e6).unstack("annee").style.format("{:,.1f}"))

# ---------------------------------------------------------------------------
# 2. Les ratios, sur les trois exercices
# ---------------------------------------------------------------------------
ratios = pd.DataFrame({
    "taux de marge": situation.marge_nette / situation.ca_net,
    "frais généraux ÷ CA": situation.frais_generaux / situation.ca_net,
    "publicité ÷ CA": situation.publicite / situation.ca_net,
    "résultat d'exploitation ÷ CA": situation.resultat_exploitation / situation.ca_net,
    "résultat net ÷ CA": situation.resultat_net / situation.ca_net,
})
print("Ratios en pourcentage du chiffre d'affaires net, par exercice :")
display(ratios.unstack("annee").style.format("{:.1%}"))
tableaux["7_situation"] = situation.unstack("annee")
tableaux["7_ratios"] = ratios.unstack("annee")

# ---------------------------------------------------------------------------
# 3. Combien de magasins sont bénéficiaires, chaque exercice ?
# ---------------------------------------------------------------------------
magasins_annee = (commun[commun.type == "Magasins"].groupby(["annee", "id_boutique", "nom_boutique"])
                  .agg(annee_complete=("annee_complete", "first"), ca_net=("ca_net", "sum"), frais_generaux=("frais_generaux", "sum"),
                       publicite=("publicite", "sum"), resultat_exploitation=("resultat_exploitation", "sum"),
                       resultat_net=("resultat_net", "sum")).reset_index())
complets = magasins_annee[magasins_annee.annee_complete]
decompte = complets.groupby("annee").agg(magasins_ouverts_toute_l_annee=("id_boutique", "size"),
                                         beneficiaires_avant_impots=("resultat_exploitation", lambda s: int((s > 0).sum())),
                                         beneficiaires_apres_impots=("resultat_net", lambda s: int((s > 0).sum())))
decompte["part des magasins bénéficiaires avant impôts"] = decompte.beneficiaires_avant_impots / decompte.magasins_ouverts_toute_l_annee
print("Magasins ouverts toute l'année : combien gagnent de l'argent ?")
display(decompte.style.format({"part des magasins bénéficiaires avant impôts": "{:.0%}"}))
tableaux["7_magasins_beneficiaires"] = decompte

# ---------------------------------------------------------------------------
# 4. Le pont : ce qui fait passer le résultat des magasins de 2019 à 2020, puis à 2021
# ---------------------------------------------------------------------------
m = situation.loc["Magasins"]
elements = [(f"Résultat {REFERENCE}", m.loc[REFERENCE, "resultat_exploitation"], "total")]
for a, b in zip(ANNEES[:-1], ANNEES[1:]):
    elements += [(f"Marge perdue {b}", m.loc[b, "marge_nette"] - m.loc[a, "marge_nette"], "écart"),
                 (f"Frais généraux {b}", -(m.loc[b, "frais_generaux"] - m.loc[a, "frais_generaux"]), "écart"),
                 (f"Publicité {b}", -(m.loc[b, "publicite"] - m.loc[a, "publicite"]), "écart"),
                 (f"Résultat {b}", m.loc[b, "resultat_exploitation"], "total")]
pont = pd.DataFrame(elements, columns=["étape", "montant", "nature"]).set_index("étape")
assert abs(pont.loc[pont.nature == "écart", "montant"].sum() + m.loc[REFERENCE, "resultat_exploitation"]
           - m.loc[EXERCICE, "resultat_exploitation"]) < 1, "Le pont ne se recoupe pas"
print("Magasins : du résultat de 2019 à celui de 2021 (+ : améliore le résultat, − : le dégrade)")
display(pont[["montant"]].style.format("{:+,.0f}"))
tableaux["7_pont_resultat_magasins"] = pont

fig, ax = plt.subplots(figsize=(11.5, 4.8))
cumul = 0.0
for x, (nom, ligne) in enumerate(pont.iterrows()):
    if ligne.nature == "total":
        bas, haut, couleur = 0, ligne.montant, viz.GRIS
        cumul = ligne.montant
    else:
        bas, haut = cumul, cumul + ligne.montant
        couleur = viz.ALERTE if ligne.montant < 0 else viz.PRINCIPALE
        cumul = haut
    ax.bar(x, haut - bas, bottom=bas, color=couleur, width=0.62)
    ax.text(x, max(bas, haut) + 2.5e6, m_dollars(ligne.montant, signe=ligne.nature != "total").replace("-", "−"),
            ha="center", fontsize=8.5, fontweight="bold", color=viz.FONCE)
ax.axhline(0, color=viz.GRIS, linewidth=1)
ax.set_xticks(range(len(pont)), [textwrap.fill(n, 11) for n in pont.index], fontsize=8.5)
niveaux = pont.montant.where(pont.nature == "total").ffill() + pont.montant.where(pont.nature == "écart").fillna(0).groupby((pont.nature == "total").cumsum()).cumsum()
ax.set_ylim(min(0, niveaux.min()) - 12e6, m.loc[REFERENCE, "resultat_exploitation"] + 20e6)
viz.format_dollars(ax, axe="y")
viz.titre(ax, f"Les magasins passent de {m.loc[REFERENCE, 'resultat_exploitation'] / 1e6:,.0f} millions de dollars de bénéfice à un déficit en deux exercices",
          f"Résultat d'exploitation des magasins, de {REFERENCE} à {EXERCICE} · vert : améliore le résultat, rouge : le dégrade")
print("Figure enregistrée :", viz.enregistrer(fig, "06_pont_resultat_magasins"))
plt.show()

**Lecture**

**L'entreprise reste bénéficiaire ; ce sont les magasins qui basculent.**

| Résultat d'exploitation | 2019 | 2020 | 2021 | En % du CA 2019 → 2021 |
|---|---|---|---|---|
| En ligne | 98,7 M$ | 105,3 M$ | 99,9 M$ | 39,2 % → 37,1 % |
| Revendeur | 70,4 M$ | 66,3 M$ | 52,6 M$ | 38,7 % → 33,7 % |
| Magasins | 121,6 M$ | 13,0 M$ | −10,8 M$ | 23,8 % → −4,1 % |
| **Ensemble** | **290,7 M$** | **184,6 M$** | **141,7 M$** | 30,7 % → 20,6 % |

Après impôts, l'entreprise gagne 187,4, puis 101,5, puis 66,9 M$. Les magasins perdent 20,4 M$ en 2020 et 39,5 M$ en 2021.

**Le taux de marge des magasins ne bouge pas (56,1 % à 56,4 %) ; ce sont les charges qui changent de poids.**

| Magasins | 2019 | 2020 | 2021 |
|---|---|---|---|
| Frais généraux | 83,9 M$ · 16,4 % du CA | 89,7 M$ · 29,3 % | 96,8 M$ · 36,8 % |
| Publicité | 81,6 M$ · 16,0 % | 70,1 M$ · 22,9 % | 61,9 M$ · 23,5 % |
| Magasins bénéficiaires avant impôts | 48 sur 48 | 43 sur 51 | 4 sur 53 |
| Magasins bénéficiaires après impôts | 48 | 5 | 1 |

Les frais généraux augmentent aussi dans les deux autres canaux : en ligne, +22,6 % en deux exercices (41,1 → 50,4 M$) pour des ventes en hausse de 6,7 % ; chez le revendeur, de 16,5 % à 21,8 % du chiffre d'affaires. La hausse des frais généraux est une décision prise pour toute l'entreprise, pas une dérive propre aux magasins.

**Le pont du résultat des magasins : ce qui relève du marché et ce qui relève des décisions.**

| | 2019 → 2020 | 2020 → 2021 | Sur deux exercices |
|---|---|---|---|
| Marge perdue (ventes) | −114,4 M$ | −24,8 M$ | −139,2 M$ |
| Hausse des frais généraux (décision) | −5,8 M$ | −7,1 M$ | −12,9 M$ |
| Baisse de la publicité (décision) | +11,6 M$ | +8,2 M$ | +19,8 M$ |
| Résultat à l'arrivée | 13,0 M$ | −10,8 M$ | |

- La baisse des ventes explique l'essentiel de la dégradation : 139,2 M$ de marge perdue, dont 114,4 M$ dès 2020.
- Les deux décisions de gestion vont en sens contraire : **réduire la publicité a amorti la perte de 19,8 M$**, **augmenter les frais généraux l'a aggravée de 12,9 M$**.
- Les hausses de frais généraux en deux exercices (12,9 M$, dont 9,1 M$ sur les 48 magasins ouverts depuis 2019 et 3,8 M$ pour les magasins ouverts en 2020) dépassent le déficit de 2021 (10,8 M$). À frais généraux de 2019, les magasins seraient encore à l'équilibre (+2,1 M$) malgré la chute de moitié des ventes.

### La publicité : ce qu'elle coûte, ce qu'on peut en dire

Chaque boutique reçoit le même budget de publicité chaque mois (`05b_charges`). On répond à trois questions :

1. **Comment le budget a-t-il évolué** sur les trois exercices, face aux ventes ?
2. **Qu'a changé sa baisse** sur le résultat des magasins ?
3. **Les ventes suivent-elles la publicité ?** Quatre tests, parce qu'on ne peut pas comparer des magasins plus ou moins soutenus : le budget est le même partout.

| Test | Si la publicité faisait vendre | Ce qu'on compare |
|---|---|---|
| A. Mois avec et sans publicité | Les mois sans publicité reculent plus que les autres quand le budget baisse | Évolution des ventes des magasins, mois de campagne et autres mois |
| B. Canaux | La boutique en ligne et le revendeur, qui reçoivent la même baisse, reculent aussi | Évolution des ventes par canal |
| C. Catégories | Les catégories reculent dans la même proportion que la publicité qui les soutient | Évolution par catégorie, face à la part de publicité de chaque catégorie |
| D. Le mois suivant | La publicité d'un mois va avec plus de ventes le mois suivant, à saison égale | Corrélation entre publicité et ventes du mois suivant, après retrait de l'effet du mois |

In [ ]:
# ---------------------------------------------------------------------------
# 1. Budget de publicité par boutique ouverte toute l'année, et ventes des magasins
# ---------------------------------------------------------------------------
pub_boutique = commun[commun.annee_complete].groupby(["annee", "id_boutique"]).publicite.sum()
budget = pd.DataFrame({
    "publicité par boutique": pub_boutique.groupby("annee").median(),
    "publicité des magasins": situation.loc["Magasins", "publicite"],
    "chiffre d'affaires des magasins": situation.loc["Magasins", "ca_net"],
})
for col in budget.columns:
    budget[f"évolution · {col}"] = budget[col].pct_change()
budget["publicité ÷ CA des magasins"] = budget["publicité des magasins"] / budget["chiffre d'affaires des magasins"]
print("Publicité et ventes des magasins, par exercice :")
display(budget.style.format({c: ("{:+.1%}" if c.startswith("évolution") else "{:.1%}" if "÷" in c else "{:,.0f}") for c in budget.columns}, na_rep=""))
tableaux["7_publicite_budget"] = budget

# ---------------------------------------------------------------------------
# 2. Ce qu'a changé la baisse : le résultat des magasins de l'exercice si la publicité était restée au niveau de 2019
# ---------------------------------------------------------------------------
pub_ref_par_boutique = budget.loc[REFERENCE, "publicité par boutique"]
pub_ex_par_boutique = budget.loc[EXERCICE, "publicité par boutique"]
nb_magasins_ex = commun[(commun.annee == EXERCICE) & (commun.type == "Magasins")].id_boutique.nunique()
economie_pub = situation.loc[("Magasins", REFERENCE), "publicite"] - situation.loc[("Magasins", EXERCICE), "publicite"]
economie_pub_bascule = situation.loc[("Magasins", BASCULE), "publicite"] - situation.loc[("Magasins", EXERCICE), "publicite"]
resultat_mag_ex = situation.loc[("Magasins", EXERCICE), "resultat_exploitation"]
print(f"Publicité des magasins : {m_dollars(economie_pub)} de moins qu'en {REFERENCE}, {m_dollars(economie_pub_bascule)} de moins qu'en {BASCULE}")
print(f"Résultat des magasins en {EXERCICE} : {m_dollars(resultat_mag_ex)} ; avec le budget de {BASCULE} : {m_dollars(resultat_mag_ex - economie_pub_bascule)} ;"
      f" avec celui de {REFERENCE} : {m_dollars(resultat_mag_ex - economie_pub)}")

# ---------------------------------------------------------------------------
# 3. Test A : mois de campagne et autres mois
# ---------------------------------------------------------------------------
mag_mois = commun[(commun.type == "Magasins") & commun.annee_complete].groupby(["annee", "mois"])[["ca_net", "publicite"]].sum()
mois_campagne = mag_mois.groupby("mois").publicite.sum() > 0
mag_mois["periode"] = mag_mois.index.get_level_values("mois").map(lambda mo: "mois avec publicité" if mois_campagne[mo] else "mois sans publicité")
test_a = mag_mois.groupby(["periode", "annee"]).ca_net.sum().unstack("annee")
test_a[f"évolution {REFERENCE} → {EXERCICE}"] = test_a[EXERCICE] / test_a[REFERENCE] - 1
test_a["mois"] = [", ".join(NOMS_MOIS[mo - 1] for mo in mois_campagne.index if (mois_campagne[mo] == (p == "mois avec publicité"))) for p in test_a.index]
print("Test A · Ventes des magasins, mois de campagne et autres mois :")
display(test_a.style.format({a: "{:,.0f}" for a in ANNEES} | {f"évolution {REFERENCE} → {EXERCICE}": "{:+.1%}"}))

# ---------------------------------------------------------------------------
# 4. Test B : la même baisse de budget par boutique, dans les trois canaux
# ---------------------------------------------------------------------------
test_b = pd.DataFrame({
    "publicité par boutique": commun[commun.annee_complete].groupby(["type", "annee"]).publicite.sum().unstack("annee").div(
        commun[commun.annee_complete].groupby(["type", "annee"]).id_boutique.nunique().unstack("annee")).pipe(lambda d: d[EXERCICE] / d[REFERENCE] - 1),
    "chiffre d'affaires": situation.ca_net.unstack("annee").pipe(lambda d: d[EXERCICE] / d[REFERENCE] - 1),
}).loc[ORDRE_TYPES]
print(f"Test B · Évolution {REFERENCE} → {EXERCICE} par canal :")
display(test_b.style.format("{:+.1%}"))

# ---------------------------------------------------------------------------
# 5. Test C : par catégorie, la part de publicité et l'évolution des ventes des magasins
# ---------------------------------------------------------------------------
pub_cat = reel[(reel.famille_compte == "Publicité") & (reel.type == "Magasins")].pivot_table(index="categorie", columns="annee", values="montant", aggfunc="sum")
ca_cat_mag = ventes[ventes.type == "Magasins"].pivot_table(index="categorie", columns="annee", values="ca_net", aggfunc="sum")
test_c = pd.DataFrame({f"part de la publicité {EXERCICE}": pub_cat[EXERCICE] / pub_cat[EXERCICE].sum(),
                       f"évolution de la publicité {REFERENCE} → {EXERCICE}": pub_cat[EXERCICE] / pub_cat[REFERENCE] - 1,
                       f"évolution des ventes {REFERENCE} → {EXERCICE}": ca_cat_mag[EXERCICE] / ca_cat_mag[REFERENCE] - 1}
                      ).sort_values(f"évolution des ventes {REFERENCE} → {EXERCICE}")
print("Test C · Magasins, par catégorie :")
display(test_c.style.format({test_c.columns[0]: "{:.1%}", test_c.columns[1]: "{:+.1%}", test_c.columns[2]: "{:+.1%}"}))
r_test_c = test_c.iloc[:, 1].corr(test_c.iloc[:, 2], method="spearman")
print(f"Corrélation de rang entre l'évolution de la publicité et celle des ventes, par catégorie : {r_test_c:+.2f}")

# ---------------------------------------------------------------------------
# 6. Test D : publicité du mois et ventes du mois suivant, à mois égal
# ---------------------------------------------------------------------------
serie = commun[(commun.type == "Magasins") & commun.annee_complete].groupby("date")[["ca_net", "publicite"]].sum().sort_index()
serie["ca_mois_suivant"] = serie.ca_net.shift(-1)
serie["mois"] = serie.index.month
for col in ["publicite", "ca_mois_suivant", "ca_net"]:
    serie[col + "_centre"] = serie[col] - serie.groupby("mois")[col].transform("mean")
r_suivant_brut = serie.publicite.corr(serie.ca_mois_suivant, method="spearman")
r_suivant_centre = serie.publicite_centre.corr(serie.ca_mois_suivant_centre, method="spearman")
print(f"Test D · Publicité du mois et ventes du mois suivant (36 mois) : {r_suivant_brut:+.2f} ; à mois égal : {r_suivant_centre:+.2f}")
tableaux["7_publicite_tests"] = pd.concat({"A": test_a.drop(columns="mois"), "B": test_b, "C": test_c}, axis=0)

**Lecture**

**Une coupe de 24 % appliquée partout de la même façon.**

| | 2019 | 2020 | 2021 |
|---|---|---|---|
| Publicité par boutique | 1,54 M$ | 1,37 M$ (−11,0 %) | 1,17 M$ (−15,0 %) |
| Publicité des magasins | 81,6 M$ | 70,1 M$ | 61,9 M$ |
| Chiffre d'affaires des magasins | 511,5 M$ | 306,2 M$ (−40,1 %) | 263,2 M$ (−14,0 %) |
| Publicité ÷ CA des magasins | 16,0 % | 22,9 % | 23,5 % |

Le budget a baissé de 24,3 % en deux exercices, **à l'identique dans chaque campagne et dans chaque canal**. Il a baissé moins vite que les ventes des magasins : son poids passe de 16,0 % à 23,5 % de leur chiffre d'affaires. La direction a réduit la publicité, mais sans la proportionner aux ventes de chaque boutique.

**Ce que la coupe a rapporté.** Avec le budget de 2020, les magasins perdraient 19,0 M$ en 2021 ; avec celui de 2019, 30,5 M$. La coupe a donc ramené leur déficit à 10,8 M$.

**Les ventes suivent-elles la publicité ? Aucun des quatre tests ne le montre.**

| Test | Résultat | Verdict |
|---|---|---|
| A. Mois avec et sans publicité | Mois de campagne (janvier à septembre) −48,0 % ; mois sans campagne (octobre à décembre) −47,4 % | Même recul : non soutenu |
| B. Canaux | Même coupe de −24,3 % par boutique ; ventes en ligne +6,7 %, revendeur −14,1 %, magasins −48,6 % | La même coupe va avec trois évolutions opposées : non soutenu |
| C. Catégories | Publicité −24,2 % dans chaque catégorie ; ventes de −72,2 % à +24,6 % ; corrélation de rang +0,14 | Non soutenu |
| D. Le mois suivant | +0,35 ; +0,66 à mois égal | Non concluant : publicité et ventes baissent toutes deux d'un exercice à l'autre, la corrélation porte cette tendance commune |

**Ce qu'on peut en conclure, et ce qu'on ne peut pas.** La baisse de 24 % du budget n'a laissé **aucune trace mesurable** sur les ventes, et elle a amélioré le résultat de 19,8 M$. Cela ne prouve pas que la publicité est inutile : chaque boutique reçoit le même budget, on ne peut donc pas comparer un magasin soutenu à un magasin qui ne l'est pas. Pour trancher, il faudrait un test piloté (des magasins témoins à budget maintenu, d'autres à budget réduit), à proposer au marketing avant d'aller plus loin dans la baisse (section 8).

### Quand les magasins perdent-ils de l'argent ? Sur quelles catégories ? Lesquels ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Magasins : résultat d'exploitation mois par mois, sur les trois exercices
# ---------------------------------------------------------------------------
mensuel = commun[commun.type == "Magasins"].groupby("date")[["ca_net", "marge_nette", "frais_generaux", "publicite", "resultat_exploitation"]].sum()
fig, ax = plt.subplots(figsize=(12, 4.4))
positions = np.arange(len(mensuel))
ax.bar(positions, mensuel.resultat_exploitation, color=np.where(mensuel.resultat_exploitation < 0, viz.ALERTE, viz.PRINCIPALE), width=0.75,
       label="Résultat d'exploitation")
ax.plot(positions, mensuel.publicite, color=viz.ACCENT, linewidth=2.2, marker="o", markersize=3.5, label="Publicité")
ax.axhline(0, color=viz.GRIS, linewidth=1)
ax.set_xticks(positions[::3], [d.strftime("%Y-%m") for d in mensuel.index[::3]], rotation=45, ha="right")
viz.format_dollars(ax, axe="y")
ax.legend(frameon=False, loc="upper right")
viz.titre(ax, "Les magasins perdent de l'argent de janvier à mars, quand tombe la publicité des Fêtes",
          "Magasins · barres : résultat d'exploitation du mois · ligne orange : dépenses de publicité")
print("Figure enregistrée :", viz.enregistrer(fig, "06_resultat_mensuel_magasins"))
plt.show()
mois_negatifs = (mensuel.resultat_exploitation < 0).groupby(mensuel.index.year).sum()
print("Mois au résultat négatif, par exercice :", mois_negatifs.to_dict())
mensuel["periode"] = np.where(mensuel.index.month.isin([mo for mo in mois_campagne.index if mois_campagne[mo]]), "mois avec publicité", "mois sans publicité")
par_periode = mensuel.groupby([mensuel.index.year, "periode"])[["ca_net", "publicite", "resultat_exploitation"]].sum()
par_periode["résultat ÷ CA"] = par_periode.resultat_exploitation / par_periode.ca_net
display(par_periode.style.format({"ca_net": "{:,.0f}", "publicite": "{:,.0f}", "resultat_exploitation": "{:,.0f}", "résultat ÷ CA": "{:.1%}"}))
tableaux["7_magasins_par_mois"] = mensuel

# ---------------------------------------------------------------------------
# 2. Résultat d'exploitation par catégorie de produit, magasins, sur chaque exercice
# ---------------------------------------------------------------------------
ventes_cat = ventes.groupby(["type", "annee", "categorie"]).agg(ca_net=("ca_net", "sum"), marge_nette=("marge_nette", "sum"))
charges_cat = (reel[reel.famille_compte.isin(["Frais généraux", "Publicité"])]
               .pivot_table(index=["type", "annee", "categorie"], columns="famille_compte", values="montant", aggfunc="sum")
               .rename(columns={"Frais généraux": "frais_generaux", "Publicité": "publicite"}))
par_cat = ventes_cat.join(charges_cat)
par_cat["resultat_exploitation"] = par_cat.marge_nette - par_cat.frais_generaux - par_cat.publicite
assert abs(par_cat.resultat_exploitation.sum() - commun.resultat_exploitation.sum()) < 1, "Le résultat par catégorie ne se recoupe pas"
cat_mag = par_cat.loc["Magasins"].resultat_exploitation.unstack("annee")
cat_mag = cat_mag.sort_values(REFERENCE, ascending=False)
print("Magasins : résultat d'exploitation par catégorie et par exercice :")
display(cat_mag.style.format("{:,.0f}").map(lambda v: f"color: {viz.ALERTE}" if v < 0 else ""))
taux_cat = (par_cat.resultat_exploitation / par_cat.ca_net).xs(EXERCICE, level="annee").unstack("type")[ORDRE_TYPES]
print(f"Résultat d'exploitation ÷ chiffre d'affaires, par catégorie et par type de boutique, {EXERCICE} :")
display(taux_cat.sort_values("Magasins", ascending=False).style.format("{:.1%}"))
tableaux["7_categories_resultat"] = par_cat.reset_index()

# ---------------------------------------------------------------------------
# 3. Les magasins bénéficiaires de l'exercice : qu'ont-ils en commun ?
# ---------------------------------------------------------------------------
m_ex = complets[complets.annee == EXERCICE].copy()
m_ex["taux_frais_generaux"] = m_ex.frais_generaux / m_ex.ca_net
ca_ref = complets[complets.annee == REFERENCE].set_index("id_boutique").ca_net
m_ex["évolution du CA depuis " + str(REFERENCE)] = m_ex.id_boutique.map(ca_ref).rdiv(m_ex.ca_net).sub(1)
m_ex["date_ouverture"] = m_ex.id_boutique.map(ouverture)
m_ex["statut"] = np.where(m_ex.resultat_exploitation > 0, "bénéficiaire", "déficitaire")
colonnes_m = ["nom_boutique", "date_ouverture", "ca_net", "évolution du CA depuis " + str(REFERENCE), "taux_frais_generaux", "resultat_exploitation"]
FORMATS_M = {"ca_net": "{:,.0f}", "évolution du CA depuis " + str(REFERENCE): "{:+.1%}", "taux_frais_generaux": "{:.1%}",
             "resultat_exploitation": "{:,.0f}", "date_ouverture": "{:%Y-%m-%d}"}
print(f"Magasins bénéficiaires en {EXERCICE} :")
display(m_ex[m_ex.statut == "bénéficiaire"].sort_values("resultat_exploitation", ascending=False)[colonnes_m]
        .style.format(FORMATS_M, na_rep="(ouvert après " + str(REFERENCE) + ")").hide(axis="index"))
print("Comparaison des deux groupes (médianes) :")
display(m_ex.groupby("statut").agg(magasins=("id_boutique", "size"), ca_net=("ca_net", "median"),
                                   evolution=("évolution du CA depuis " + str(REFERENCE), "median"),
                                   taux_frais_generaux=("taux_frais_generaux", "median"))
        .style.format({"ca_net": "{:,.0f}", "evolution": "{:+.1%}", "taux_frais_generaux": "{:.1%}"}))
tableaux["7_magasins_exercice"] = m_ex

**Lecture**

**Les magasins perdent de l'argent de janvier à mars, chaque exercice.** Les trois mois au résultat négatif sont, les trois années, janvier, février et mars : en 2021, −12,4, −15,3 et −15,8 M$. C'est le trimestre où est engagée la publicité des Fêtes (44,5 M$ en 2021, soit 69 % du budget, section 8), alors que janvier est aussi le mois le plus faible en ventes (section 3).

| Résultat ÷ CA des magasins | 2019 | 2020 | 2021 |
|---|---|---|---|
| Mois avec publicité (janvier à septembre) | 19,3 % | −1,6 % | −10,6 % |
| Mois sans publicité (octobre à décembre) | 38,4 % | 24,4 % | 17,6 % |

Même en 2021, un magasin reste bénéficiaire d'octobre à décembre (+10,7 M$ au total) ; il perd 21,4 M$ sur les neuf mois de campagne.

*Question à poser au marketing* : la publicité « des Fêtes » est-elle engagée après les fêtes (règlement décalé des factures de la campagne de fin d'année), ou la campagne est-elle réellement diffusée de janvier à mars ? Dans le second cas, le calendrier des campagnes est à revoir : le quatrième trimestre ne reçoit aucune publicité.

**Par catégorie : ce n'est pas la catégorie qui perd, c'est le magasin.**

| Magasins | 2019 | 2020 | 2021 |
|---|---|---|---|
| Appareils photo | +43,6 M$ | +5,2 M$ | −8,7 M$ |
| Ordinateurs | +27,9 M$ | +1,9 M$ | +14,3 M$ |
| Électroménager | +27,5 M$ | +4,6 M$ | −11,8 M$ |
| Téléphones | +12,3 M$ | −0,5 M$ | −1,9 M$ |
| Jeux et jouets | −1,0 M$ | −1,5 M$ | −1,6 M$ |

- Les appareils photo, première source de résultat des magasins en 2019, perdent 52,3 M$ de résultat en deux exercices.
- Les ordinateurs restent la seule grande catégorie bénéficiaire en magasin en 2021 (14,5 % du chiffre d'affaires).
- En ligne et chez le revendeur, **toutes les catégories gagnent** entre 27 % et 46 % de leur chiffre d'affaires. La même catégorie qui gagne 35 % en ligne perd 21 % en magasin (appareils photo) : le problème est le coût du magasin rapporté à ce qu'il vend, pas le produit.
- Les jeux et jouets sont déficitaires en magasin les trois exercices. Piste à étudier : les réserver à la vente en ligne, où ils font 3,9 % du chiffre d'affaires et 45,5 % de résultat.

**Les quatre magasins bénéficiaires de 2021.**

| Magasin | Ouverture | CA 2021 | Depuis 2019 | Frais généraux ÷ CA | Résultat |
|---|---|---|---|---|---|
| Moscow No.2 | 2016 | 13,08 M$ | +23,6 % | 18,5 % | +3,79 M$ |
| Cambridge | 2016 | 5,57 M$ | −39,8 % | 28,1 % | +0,37 M$ |
| Berlin | octobre 2020 | 3,71 M$ | | 17,9 % | +0,26 M$ |
| Marseille | avril 2020 | 3,60 M$ | | 21,6 % | +0,10 M$ |

Les bénéficiaires ne sont pas plus gros : leur chiffre d'affaires médian (4,64 M$) est même un peu inférieur à celui des déficitaires (4,84 M$). Ce qui les distingue est le **poids des frais généraux** (médiane 20,0 % contre 39,0 %), obtenu par deux voies :

- **des ventes qui ont résisté** : Moscow No.2 et Cambridge, dont les ventes ont moins baissé que celles des autres magasins (médiane −8,1 % contre −53,7 %), si bien que les frais généraux y pèsent moins ;
- **des charges à la mesure des ventes** : Berlin et Marseille, ouverts en 2020, dont les frais généraux sont fixés sur leur activité actuelle et non reconduits depuis 2019 (hypothèse : la base de ces magasins est leur année d'ouverture).

Ce sont précisément les deux familles de leviers chiffrées en section 8 : regagner des ventes, et recaler les charges sur les ventes.

## 8. Les leviers sur l'exercice 2021

On passe du constat à l'action. Chaque levier est chiffré sur le dernier exercice : c'est la situation à régler. Avant de chiffrer, il faut savoir ce que coûte un dollar de vente perdu, et donc quelles charges suivent les ventes.

**Ce que coûte un dollar de vente.** Quand un magasin vend un dollar de moins, il perd sa marge (environ 56 cents), mais il ne paie plus les charges qui suivent les ventes. Avant impôts, ce sont les « autres charges », environ 1 cent. Les autres frais généraux (personnel, loyers, administration, informatique, énergie) et la publicité ne baissent pas : ils sont décidés par la direction, exercice par exercice. Un dollar de vente rapporte donc environ 55 cents de résultat d'exploitation.

In [ ]:
# ---------------------------------------------------------------------------
# 0. Ce que rapporte un dollar de vente : marge moins charges qui suivent les ventes
# ---------------------------------------------------------------------------
commun_ex = commun[commun.annee == EXERCICE]
reel_ex = reel[reel.annee == EXERCICE]
# Le résultat d'exploitation est calculé avant impôts : seule la charge "Autres charges" suit les ventes à ce niveau
COMPTES_VARIABLES = ["Autres charges"]
variables_ex = reel_ex[reel_ex.nom_compte.isin(COMPTES_VARIABLES)].groupby("type").montant.sum()
contribution = pd.DataFrame({"chiffre d'affaires net": commun_ex.groupby("type").ca_net.sum(),
                             "marge nette": commun_ex.groupby("type").marge_nette.sum(),
                             "charges qui suivent les ventes": variables_ex}).loc[ORDRE_TYPES]
contribution["ce que rapporte 1 $ de vente"] = (contribution["marge nette"] - contribution["charges qui suivent les ventes"]) / contribution["chiffre d'affaires net"]
display(contribution.style.format({"chiffre d'affaires net": "{:,.0f}", "marge nette": "{:,.0f}", "charges qui suivent les ventes": "{:,.0f}",
                                   "ce que rapporte 1 $ de vente": "{:.3f}"}))
TAUX_CONTRIBUTION = float((contribution["marge nette"].sum() - contribution["charges qui suivent les ventes"].sum()) / contribution["chiffre d'affaires net"].sum())
TAUX_CONTRIBUTION_MAGASINS = float(contribution.loc["Magasins", "ce que rapporte 1 $ de vente"])
print(f"Un dollar de vente rapporte {TAUX_CONTRIBUTION:.3f} $ de résultat (magasins : {TAUX_CONTRIBUTION_MAGASINS:.3f} $)")

# ---------------------------------------------------------------------------
# 1. Seuil de rentabilité de chaque magasin, sur l'exercice
# ---------------------------------------------------------------------------
# Le magasin est à l'équilibre quand ce que rapportent ses ventes couvre ses charges fixes :
#   seuil = (frais généraux fixes + publicité) ÷ ce que rapporte 1 $ de vente
fg_fixes_ex = (reel_ex[(reel_ex.type == "Magasins") & (reel_ex.famille_compte == "Frais généraux") & ~reel_ex.nom_compte.isin(COMPTES_VARIABLES)]
               .groupby("id_boutique").montant.sum())
seuils_mag = m_ex.set_index("id_boutique").copy()
seuils_mag["charges fixes"] = fg_fixes_ex + seuils_mag.publicite
variables_magasin = reel_ex[(reel_ex.type == "Magasins") & reel_ex.nom_compte.isin(COMPTES_VARIABLES)].groupby("id_boutique").montant.sum()
marge_magasin = commun_ex[commun_ex.type == "Magasins"].groupby("id_boutique").marge_nette.sum()
seuils_mag["ce que rapporte 1 $ de vente"] = (marge_magasin - variables_magasin) / seuils_mag.ca_net
seuils_mag["seuil de rentabilité"] = seuils_mag["charges fixes"] / seuils_mag["ce que rapporte 1 $ de vente"]
seuils_mag["CA ÷ seuil"] = seuils_mag.ca_net / seuils_mag["seuil de rentabilité"]
resume_seuil = seuils_mag.groupby("statut").agg(magasins=("nom_boutique", "size"), ca_net=("ca_net", "median"),
                                                seuil=("seuil de rentabilité", "median"), rapport=("CA ÷ seuil", "median"))
resume_seuil["ventes manquantes (médiane)"] = (resume_seuil.seuil - resume_seuil.ca_net).clip(lower=0)
print(f"Seuil de rentabilité des magasins ouverts toute l'année, {EXERCICE} (médianes) :")
display(resume_seuil.style.format({"ca_net": "{:,.0f}", "seuil": "{:,.0f}", "rapport": "{:.2f}", "ventes manquantes (médiane)": "{:,.0f}"}))
deficitaires = seuils_mag[seuils_mag.statut == "déficitaire"]
print(f"Un magasin déficitaire doit vendre en médiane {(deficitaires['seuil de rentabilité'] / deficitaires.ca_net - 1).median():+.1%} pour atteindre l'équilibre")
tableaux["8_seuils"] = seuils_mag

# ---------------------------------------------------------------------------
# 2. Même budget de publicité, réparti en proportion du chiffre d'affaires
# ---------------------------------------------------------------------------
boutiques_ex = commun_ex.groupby(["id_boutique", "type"])[["ca_net", "marge_nette", "frais_generaux", "publicite", "resultat_exploitation"]].sum().reset_index()
budget_total = boutiques_ex.publicite.sum()
boutiques_ex["publicité proportionnelle"] = budget_total * boutiques_ex.ca_net / boutiques_ex.ca_net.sum()
boutiques_ex["résultat simulé"] = boutiques_ex.marge_nette - boutiques_ex.frais_generaux - boutiques_ex["publicité proportionnelle"]
repartition = boutiques_ex.groupby("type").agg(
    boutiques=("id_boutique", "size"), publicite_actuelle=("publicite", "sum"), publicite_proportionnelle=("publicité proportionnelle", "sum"),
    resultat_actuel=("resultat_exploitation", "sum"), resultat_simule=("résultat simulé", "sum"),
    deficitaires_actuel=("resultat_exploitation", lambda s: int((s < 0).sum())),
    deficitaires_simule=("résultat simulé", lambda s: int((s < 0).sum()))).loc[ORDRE_TYPES]
repartition.loc["Ensemble"] = repartition.sum()
print(f"Répartition du même budget de publicité en proportion du chiffre d'affaires ({budget_total / boutiques_ex.ca_net.sum():.1%} des ventes de chaque boutique), {EXERCICE} :")
display(repartition.style.format("{:,.0f}"))
tableaux["8_repartition_publicite"] = repartition

**Lecture**

**Un dollar de vente rapporte 55 cents de résultat.** La marge nette des retours (56,2 % en magasin) moins les « autres charges », qui suivent les ventes (1,1 %), donne 0,552 $ par dollar vendu (0,551 $ en magasin). Toute vente perdue coûte 55 cents de résultat, toute vente regagnée en rapporte autant, parce que les autres charges ne bougent pas avec les ventes.

**Les magasins déficitaires sont proches de l'équilibre.**

| 2021, médianes | Magasins | Chiffre d'affaires | Seuil de rentabilité | CA ÷ seuil | Ventes manquantes |
|---|---|---|---|---|---|
| Bénéficiaires | 4 | 4,64 M$ | 3,43 M$ | 1,15 | aucune |
| Déficitaires | 49 | 4,84 M$ | 5,46 M$ | 0,89 | 0,62 M$ |

Un magasin déficitaire doit vendre **12,5 % de plus** pour atteindre l'équilibre, soit moins que ce qu'il a perdu sur le seul exercice 2021 (−14,9 %). Le déficit n'est pas celui de magasins trop petits : c'est celui de charges dimensionnées sur les ventes de 2019, avec deux fois moins de ventes.

**La règle de répartition de la publicité fait le déficit des magasins.**

| 2021 | Publicité actuelle | Publicité proportionnelle (9,3 % des ventes) | Résultat actuel | Résultat simulé | Magasins déficitaires |
|---|---|---|---|---|---|
| En ligne | 1,2 M$ | 25,1 M$ | 99,9 M$ | 75,9 M$ | |
| Revendeur | 1,2 M$ | 14,6 M$ | 52,6 M$ | 39,2 M$ | |
| Magasins | 61,9 M$ | 24,6 M$ | −10,8 M$ | +26,5 M$ | 49 → 0 |
| Ensemble | 64,2 M$ | 64,2 M$ | 141,7 M$ | 141,7 M$ | |

Chaque boutique reçoit aujourd'hui le même budget (1,17 M$), alors que la boutique en ligne vend 55 fois plus qu'un magasin. Répartir le **même budget** en proportion des ventes ne change rien au résultat de l'entreprise, mais aucun magasin ne serait plus déficitaire.

**Ce que cela change pour la décision** : ce n'est pas un gain, c'est une question d'imputation. Mais elle conditionne tout jugement sur un magasin : avant d'envisager de fermer un magasin « déficitaire », il faut lui imputer la publicité qui le sert réellement. Une campagne de presse ou internet profite aussi à la boutique en ligne.

### Les charges, compte par compte, sur les trois exercices

Le plan donne **six comptes** de frais généraux et **huit comptes** de publicité. On regarde ce que pèse chacun, comment il a évolué depuis 2019, et donc quelles décisions de gestion ont été prises pendant que les ventes baissaient.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Magasins ouverts depuis 2019 : ventes et charges, compte par compte, sur les trois exercices
# ---------------------------------------------------------------------------
nos_magasins = commun.loc[commun.type == "Magasins", "id_boutique"].unique()
depuis_ref = ouverture[(ouverture < pd.Timestamp(REFERENCE, 1, 2)) & ouverture.index.isin(nos_magasins)].index
reel_mag = reel[(reel.type == "Magasins") & reel.id_boutique.isin(depuis_ref)]
COMPTES_SUIVIS = ["Ventes", "Personnel", "Administration", "Informatique", "Loyers et immobilier", "Énergie et communication",
                  "Autres charges", "Impôts et taxes"]
COMPTES_MAITRISABLES = ["Personnel", "Administration", "Informatique"]    # décidés par l'entreprise ; loyers et énergie sont subis
annuel = (reel_mag[reel_mag.nom_compte.isin(COMPTES_SUIVIS)]
          .pivot_table(index="nom_compte", columns="annee", values="montant", aggfunc="sum").reindex(COMPTES_SUIVIS))
for a, b in zip(ANNEES[:-1], ANNEES[1:]):
    annuel[f"{a} → {b}"] = annuel[b] / annuel[a] - 1
for a in ANNEES:
    annuel[f"part des ventes {a}"] = annuel[a] / annuel.loc["Ventes", a]
print(f"Les {len(depuis_ref)} magasins ouverts depuis {REFERENCE} : ventes et charges, compte par compte")
display(annuel.style.format({a: "{:,.0f}" for a in ANNEES} | {c: "{:+.1%}" for c in annuel.columns if "→" in str(c)}
                            | {f"part des ventes {a}": "{:.1%}" for a in ANNEES}))
tableaux["8_charges_par_compte"] = annuel

# Toutes les boutiques sont-elles concernées par la hausse ?
par_boutique_compte = (reel[reel.nom_compte.isin(COMPTES_MAITRISABLES) & reel.id_boutique.isin(depuis_ref)]
                       .pivot_table(index="id_boutique", columns="annee", values="montant", aggfunc="sum"))
print(f"Magasins dont le personnel, l'administration et l'informatique augmentent de {REFERENCE} à {EXERCICE} :",
      int((par_boutique_compte[EXERCICE] > par_boutique_compte[REFERENCE]).sum()), "sur", len(par_boutique_compte))

# ---------------------------------------------------------------------------
# 2. Publicité, campagne par campagne, et calendrier
# ---------------------------------------------------------------------------
pub_compte = reel[reel.famille_compte == "Publicité"].pivot_table(index="nom_compte", columns="annee", values="montant", aggfunc="sum")
pub_compte = pub_compte.sort_values(EXERCICE, ascending=False)
pub_compte.loc["Total"] = pub_compte.sum()
pub_compte[f"évolution {REFERENCE} → {EXERCICE}"] = pub_compte[EXERCICE] / pub_compte[REFERENCE] - 1
print("Publicité par campagne, toutes boutiques :")
display(pub_compte.style.format({a: "{:,.0f}" for a in ANNEES} | {f"évolution {REFERENCE} → {EXERCICE}": "{:+.1%}"}))
reel["campagne"] = np.where(reel.nom_compte.str.startswith("Publicité des Fêtes"), "Publicité des Fêtes", reel.nom_compte)
calendrier = (reel[(reel.famille_compte == "Publicité") & (reel.annee == EXERCICE)]
              .pivot_table(index="campagne", columns="mois", values="montant", aggfunc="sum").reindex(columns=range(1, 13)).fillna(0))
calendrier.columns = NOMS_MOIS
calendrier["Total"] = calendrier.sum(axis=1)
print(f"Calendrier des campagnes en {EXERCICE}, toutes boutiques (millions de dollars) :")
display((calendrier.sort_values("Total", ascending=False) / 1e6).style.format(lambda v: "" if v == 0 else f"{v:,.1f}"))
tableaux["8_publicite_par_campagne"] = pub_compte
tableaux["8_calendrier_publicite"] = calendrier

**Lecture**

**Les frais généraux : des budgets reconduits et augmentés pendant que les ventes se divisaient par deux.** Sur les 48 magasins ouverts depuis 2019 :

| Compte | 2019 | 2021 | 2019 → 2020 | 2020 → 2021 | Part des ventes 2019 → 2021 | Nature |
|---|---|---|---|---|---|---|
| Ventes | 509,8 M$ | 245,4 M$ | −43,4 % | −14,9 % | | |
| Personnel | 22,0 M$ | 25,8 M$ | +8,4 % | +8,1 % | 4,3 % → 10,5 % | décidé |
| Administration | 11,0 M$ | 14,6 M$ | +15,3 % | +15,0 % | 2,2 % → 5,9 % | décidé |
| Informatique | 11,0 M$ | 12,9 M$ | +8,4 % | +8,1 % | 2,2 % → 5,2 % | décidé |
| Loyers et immobilier | 22,0 M$ | 22,8 M$ | +0,5 % | +3,2 % | 4,3 % → 9,3 % | engagé (baux) |
| Énergie et communication | 11,0 M$ | 12,8 M$ | +8,5 % | +8,0 % | 2,1 % → 5,2 % | subi (prix) |
| Autres charges | 5,4 M$ | 2,6 M$ | −43,9 % | −14,5 % | 1,1 % | suit les ventes |
| Impôts et taxes | 54,9 M$ | 26,4 M$ | −43,5 % | −14,8 % | 10,8 % | suit les ventes |

- **Personnel, administration et informatique augmentent dans les 48 magasins, sans exception.** Ce sont des décisions de budget, prises chaque exercice, et elles ont été prises à la hausse alors que les ventes reculaient de 43 % puis de 15 %. L'administration, à +15 % par an, est le compte qui progresse le plus.
- Les loyers sont presque stables : ils sont engagés par des baux et ne se renégocient pas en un exercice.
- L'énergie progresse de 8 % par an : c'est une dépense subie, liée aux prix, sur laquelle la marge de manœuvre est faible.
- Seuls les autres charges et les impôts suivent les ventes.

**La publicité : une coupe uniforme, et une campagne des Fêtes engagée en début d'année.**

| Campagne | 2021 | 2019 → 2021 | Mois d'engagement en 2021 |
|---|---|---|---|
| Fêtes (presse, internet, radio et télévision, autres médias) | 44,5 M$ | −24,3 % | janvier à mars |
| Rentrée | 12,0 M$ | −23,9 % | janvier à avril, août et septembre |
| Printemps | 4,4 M$ | −24,3 % | janvier à avril |
| Entreprises | 2,3 M$ | −24,3 % | janvier à mars |
| Été | 1,0 M$ | −24,3 % | mai à juillet |
| **Total** | **64,2 M$** | **−24,2 %** | aucune dépense d'octobre à décembre |

Toutes les campagnes ont reçu la même coupe : la direction n'a pas choisi entre les campagnes. Avec une mesure de l'efficacité de chacune (test piloté, section 7), la baisse pourrait porter d'abord sur les moins utiles.

### Les leviers, chiffrés sur l'exercice 2021

Chaque levier est chiffré en gain de résultat d'exploitation par an. Pour ceux qui risquent de faire baisser les ventes, on donne un **seuil** : la baisse des ventes qu'on peut supporter en restant au moins aussi rentable qu'aujourd'hui.

In [ ]:
resultat_an = commun_ex.resultat_exploitation.sum()
magasins_ex_c = commun_ex[commun_ex.type == "Magasins"]
resultat_magasins_an = magasins_ex_c.resultat_exploitation.sum()
ca_magasins_an = magasins_ex_c.ca_net.sum()
publicite_magasins_an = magasins_ex_c.publicite.sum()
leviers = []


def ajouter_levier(famille, nom, gain, commentaire, ventes_exposees=None, magasins=True):
    """Ajoute un levier : gain annuel, résultat après, et seuil s'il peut faire baisser les ventes.

    ventes_exposees : chiffre d'affaires annuel qui pourrait réagir au levier (None : pas de seuil).
    magasins        : True si le gain revient entièrement aux magasins (on donne alors leur résultat après).
    Seuil = gain ÷ (ventes exposées × ce que rapporte 1 $ de vente).
    """
    leviers.append({"famille": famille, "levier": nom, "gain par an": gain,
                    "résultat d'exploitation après": resultat_an + gain,
                    "résultat des magasins après": resultat_magasins_an + gain if magasins else np.nan,
                    "hausse du résultat": gain / resultat_an,
                    "baisse des ventes qui annule le gain": np.nan if ventes_exposees is None else gain / (ventes_exposees * TAUX_CONTRIBUTION_MAGASINS),
                    "commentaire": commentaire})


# ---------------------------------------------------------------------------
# a) Frais généraux : revenir sur les hausses décidées depuis 2019
# ---------------------------------------------------------------------------
gain_fg_montant = float((annuel.loc[COMPTES_MAITRISABLES, EXERCICE] - annuel.loc[COMPTES_MAITRISABLES, REFERENCE]).sum())
gain_fg_bascule = float((annuel.loc[COMPTES_MAITRISABLES, EXERCICE] - annuel.loc[COMPTES_MAITRISABLES, BASCULE]).sum())
gain_fg_part_ref = float((annuel.loc[COMPTES_MAITRISABLES, EXERCICE]
                          - annuel.loc[COMPTES_MAITRISABLES, f"part des ventes {REFERENCE}"] * annuel.loc["Ventes", EXERCICE]).sum())
ajouter_levier("Charges", f"Ramener personnel, administration et informatique au montant de {REFERENCE}", gain_fg_montant,
               f"Annule les hausses décidées depuis {REFERENCE} ; {len(depuis_ref)} magasins ouverts depuis {REFERENCE}")
ajouter_levier("Charges", f"Ramener les mêmes comptes au montant de {BASCULE}", gain_fg_bascule,
               "Première étape : annuler la hausse du dernier exercice")
print(f"Plafond théorique : les mêmes comptes à leur part des ventes de {REFERENCE} rapporteraient {m_dollars(gain_fg_part_ref)}, "
      f"soit une baisse de {gain_fg_part_ref / annuel.loc[COMPTES_MAITRISABLES, EXERCICE].sum():.0%} de ces charges : hors de portée en un exercice")

# ---------------------------------------------------------------------------
# b) Regrouper l'administration et l'informatique par pays (piste de réflexion)
# ---------------------------------------------------------------------------
# Principe : dans un pays qui compte plusieurs magasins, un seul bureau au lieu d'un par magasin.
# Les données ne permettent pas d'isoler la part fixe d'un bureau : le coût de l'administration et de
# l'informatique d'un magasin dépend de son niveau de 2019, reconduit, et non de son activité actuelle
# (corrélation calculée ci-dessous). On donne donc une fourchette : l'économie est une part du poste
# des pays qui comptent plusieurs magasins, part qu'il revient aux gens du métier de fixer.
reel_mag_ex = reel_ex[(reel_ex.type == "Magasins") & reel_ex.id_boutique.isin(m_ex.id_boutique)]
bureau = reel_mag_ex.pivot_table(index="id_boutique", columns="nom_compte", values="montant", aggfunc="sum")
bureau = bureau.join(dim_boutique.set_index("id_boutique")[["nom_boutique", "pays"]])
bureau["bureau_par_an"] = bureau["Administration"] + bureau["Informatique"]
ventes_ref_magasin = reel[(reel.annee == REFERENCE) & (reel.nom_compte == "Ventes")].groupby("id_boutique").montant.sum()
print(f"Corrélation, entre magasins, du coût du bureau ({EXERCICE}) avec les ventes de {EXERCICE} : "
      f"{bureau.bureau_par_an.corr(bureau['Ventes']):+.2f} · avec les ventes de {REFERENCE} : "
      f"{bureau.bureau_par_an.corr(bureau.index.to_series().map(ventes_ref_magasin)):+.2f}")
regroupement = bureau.groupby("pays").agg(magasins=("bureau_par_an", "size"), bureau_actuel=("bureau_par_an", "sum"))
regroupement = regroupement[regroupement.magasins > 1].sort_values("magasins", ascending=False)
PARTS_ECONOMISEES = [0.20, 0.30, 0.40]
for part in PARTS_ECONOMISEES:
    regroupement[f"économie si {part:.0%} du poste".replace("%", " %")] = regroupement.bureau_actuel * part
regroupement.loc["Total"] = regroupement.sum()
print(f"Regroupement de l'administration et de l'informatique par pays (pays à plusieurs magasins), {EXERCICE} :")
display(regroupement.style.format({"magasins": "{:.0f}"} | {c: "{:,.0f}" for c in regroupement.columns if c != "magasins"}))
tableaux["8_regroupement_par_pays"] = regroupement
gain_regroupement = float(regroupement.loc["Total", "bureau_actuel"] * 0.30)
ajouter_levier("Charges", "Regrouper administration et informatique par pays (hypothèse : 30 % du poste)", gain_regroupement,
               "Fourchette de 20 à 40 % ; recoupe en partie le retour au montant de " + str(REFERENCE) + ", ne pas additionner")

# ---------------------------------------------------------------------------
# c) Publicité des magasins
# ---------------------------------------------------------------------------
simulation_pub = []
for baisse in [0, 0.05, 0.10, 0.20]:
    resultat = boutiques_ex.resultat_exploitation + baisse * boutiques_ex.publicite * (boutiques_ex.type == "Magasins")
    simulation_pub.append({"baisse de la publicité des magasins": baisse, "résultat des magasins": resultat[boutiques_ex.type == "Magasins"].sum(),
                           "magasins déficitaires": int((resultat[boutiques_ex.type == "Magasins"] < 0).sum())})
    if baisse > 0:
        ajouter_levier("Publicité", f"Baisser la publicité des magasins de {baisse:.0%}".replace("%", " %"), baisse * publicite_magasins_an,
                       "Aucun lien mesurable entre publicité et ventes (section 7)", ventes_exposees=ca_magasins_an)
simulation_pub = pd.DataFrame(simulation_pub).set_index("baisse de la publicité des magasins")
print("Baisse de la publicité des magasins, à ventes inchangées :")
display(simulation_pub.style.format({"résultat des magasins": "{:,.0f}"}).format_index("{:.0%}"))
tableaux["8_baisse_publicite"] = simulation_pub

# ---------------------------------------------------------------------------
# d) Remises en magasin, retours, gamme
# ---------------------------------------------------------------------------
# Une remise supprimée ou un retour évité laisse la vente dans le chiffre d'affaires : les « autres charges »,
# qui suivent les ventes, s'appliquent à ce montant. Le gain est donc net de ces charges (même calcul que 07, 08 et 09).
part_autres_charges = (contribution["charges qui suivent les ventes"] / contribution["chiffre d'affaires net"]).to_dict()
part_autres_charges["Ensemble"] = float(contribution["charges qui suivent les ventes"].sum() / contribution["chiffre d'affaires net"].sum())
remises_ex = ventes_ex.groupby(["type", "taux_remise_annonce"]).montant_remise.sum()
remises_magasins = remises_ex.loc["Magasins"].sum()
gain_remises_magasins = remises_magasins * (1 - part_autres_charges["Magasins"])
ajouter_levier("Promotions", "Supprimer les remises en magasin", gain_remises_magasins,
               "En magasin, une ligne remisée ne contient pas plus d'unités qu'une autre (09_q5_remises)")
gain_promo_20 = remises_ex.loc[("Magasins", 0.20)] * (1 - part_autres_charges["Magasins"])
ajouter_levier("Promotions", "dont : suspendre la promotion à 20 % en magasin (octobre à janvier)", gain_promo_20,
               "Partie du levier précédent, ne pas additionner (07_q2_promotions)")
gain_plafond = sum(remises_ex.loc[(t, 0.20)] * 0.5 * (1 - part_autres_charges[t]) for t in ["En ligne", "Revendeur"])
ajouter_levier("Promotions", "Plafonner la remise à 10 % en ligne et chez le revendeur (à tester avec un groupe témoin)", gain_plafond,
               "Lecture prudente ; le lien entre remise et taille des lignes y reste ouvert (09_q5_remises)", magasins=False)
retours_ex = ventes_ex.groupby("type")[["marge_perdue_retours", "montant_retour"]].sum()
gain_retours = sum((retours_ex.loc[t, "marge_perdue_retours"] - part_autres_charges[t] * retours_ex.loc[t, "montant_retour"]) / 2 for t in ORDRE_TYPES)
ajouter_levier("Retours", "Réduire les retours de moitié", gain_retours,
               "Fréquence uniforme, aucune cause identifiable ; coût concentré sur les produits chers (08_q3_q4_retours)", magasins=False)
print(f"Remises en magasin : {m_dollars(remises_magasins, 2)} de remises, gain net {m_dollars(gain_remises_magasins, 2)} · "
      f"retours : marge perdue {m_dollars(retours_ex.marge_perdue_retours.sum(), 2)}, gain net de la moitié {m_dollars(gain_retours, 2)}")
gammes = ventes_ex.groupby("classe")[["marge_nette", "ca_net"]].sum()
taux_gamme = gammes.marge_nette / gammes.ca_net
ajouter_levier("Ventes", "Déplacer 1 % du chiffre d'affaires de la gamme économique vers la gamme luxe",
               0.01 * ventes_ex.ca_net.sum() * (taux_gamme["Deluxe"] - taux_gamme["Economy"]),
               "Va contre la descente en gamme observée (06b)", magasins=False)

### Renouveler l'assortiment : ce que rapporte la stratégie commune aux deux branches

`06b` conclut qu'il faut renouveler l'assortiment, que le produit non vendu dorme en rayon ou qu'il ne soit plus proposé. On en chiffre trois volets :

| Volet | Calcul | Hypothèse |
|---|---|---|
| 1. Revenir au rythme de lancement de 2019 | Nouveautés supplémentaires × chiffre d'affaires d'une nouveauté de l'année en 2021 × ce que rapporte 1 $ de vente | Une nouveauté de plus se vend comme les nouveautés de 2021 ; le coût de lancement n'est pas dans les données |
| 2. Remettre les nouveautés dans les magasins | Chiffre d'affaires des nouveautés en magasin si leur rapport aux ventes en ligne était celui des produits anciens | Les magasins vendent une nouveauté aussi bien qu'un produit ancien, par rapport à la boutique en ligne |
| 3. Libérer le stock dormant | Couples (magasin, produit) sans vente en 2021 × stock moyen d'un produit par relevé × coût unitaire | Le stock n'est pas relevé partout : valeur estimée. C'est de la **trésorerie** récupérée une fois, pas un gain de résultat annuel |

In [ ]:
# ---------------------------------------------------------------------------
# Volet 1 : revenir au rythme de lancement de l'année de référence
# ---------------------------------------------------------------------------
lancement = dim_produit.set_index("id_produit").date_mise_en_vente.dt.year
ventes["nouveaute_de_l_annee"] = ventes.annee <= ventes.id_produit.map(lancement)
lances = dim_produit[dim_produit.id_produit.isin(ventes.id_produit.unique())].date_mise_en_vente.dt.year.value_counts()
nouveautes_ex = ventes[(ventes.annee == EXERCICE) & ventes.nouveaute_de_l_annee]
ca_par_nouveaute = nouveautes_ex.ca_net.sum() / lances[EXERCICE]
nouveautes_en_plus = lances[REFERENCE] - lances[EXERCICE]
gain_rythme = nouveautes_en_plus * ca_par_nouveaute * TAUX_CONTRIBUTION
texte_ca = f"{ca_par_nouveaute:,.0f} $".replace(",", " ")
print(f"Lancements : {lances[REFERENCE]} en {REFERENCE} · {lances[EXERCICE]} en {EXERCICE} · chiffre d'affaires d'une nouveauté de l'année en {EXERCICE} : {texte_ca}")
print(f"Volet 1 : {nouveautes_en_plus} nouveautés de plus × {texte_ca} × {TAUX_CONTRIBUTION:.3f} = {m_dollars(gain_rythme)} de résultat par an")
ajouter_levier("Assortiment", f"Revenir au rythme de lancement de {REFERENCE} ({lances[REFERENCE]} nouveautés par an au lieu de {lances[EXERCICE]})",
               gain_rythme, "Gain brut : le coût de conception et de lancement n'est pas dans les données", magasins=False)

# ---------------------------------------------------------------------------
# Volet 2 : remettre les nouveautés dans les magasins
# ---------------------------------------------------------------------------
age = np.where(ventes.nouveaute_de_l_annee, "nouveauté de l'année", "produit plus ancien")
ca_age = ventes[ventes.annee == EXERCICE].assign(age=age[ventes.annee == EXERCICE]).pivot_table(index="age", columns="type", values="ca_net", aggfunc="sum")
rapport_age = ca_age["Magasins"] / ca_age["En ligne"]
ca_nouveautes_cible = ca_age.loc["nouveauté de l'année", "En ligne"] * rapport_age["produit plus ancien"]
gain_nouveautes_magasins = (ca_nouveautes_cible - ca_age.loc["nouveauté de l'année", "Magasins"]) * TAUX_CONTRIBUTION_MAGASINS
NOUVEAUTE, ANCIEN = "nouveauté de l'année", "produit plus ancien"
print(f"Magasins ÷ en ligne, {EXERCICE} : nouveautés {rapport_age[NOUVEAUTE]:.2f}, produits plus anciens {rapport_age[ANCIEN]:.2f}")
print(f"Volet 2 : chiffre d'affaires des nouveautés en magasin {m_dollars(ca_age.loc[NOUVEAUTE, 'Magasins'])} → "
      f"{m_dollars(ca_nouveautes_cible)} · gain de résultat {m_dollars(gain_nouveautes_magasins)}")
ajouter_levier("Assortiment", "Vendre les nouveautés en magasin aussi bien que les produits anciens", gain_nouveautes_magasins,
               "Mise en rayon et mise en avant des nouveautés dans tous les magasins")

# ---------------------------------------------------------------------------
# Volet 3 : le stock dormant (trésorerie, hors classement des gains annuels)
# ---------------------------------------------------------------------------
stock = pd.read_parquet(chemins["stock"], columns=["annee", "id_boutique", "id_produit", "quantite_en_stock", "cout_unitaire"])
stock_mag_ex = stock[(stock.annee == EXERCICE) & stock.id_boutique.isin(m_ex.id_boutique)]
valeur_par_produit = (stock_mag_ex.quantite_en_stock * stock_mag_ex.cout_unitaire).groupby(stock_mag_ex.id_produit).mean()
vendus_ex = magasins_lignes[magasins_lignes.annee == EXERCICE].groupby(["id_boutique", "id_produit"]).size()
produits_ex = ventes_ex[ventes_ex.type == "Magasins"].id_produit.unique()
grille = pd.MultiIndex.from_product([m_ex.id_boutique.unique(), produits_ex], names=["id_boutique", "id_produit"])
sans_vente = grille.difference(vendus_ex.index)
valeurs = pd.Series(sans_vente.get_level_values("id_produit")).map(valeur_par_produit)
stock_dormant = float(valeurs.sum())
print(f"Couples (magasin × produit) sans vente en {EXERCICE} : {len(sans_vente):,} sur {len(grille):,}".replace(",", " "),
      f"· valeur estimée du stock dormant : {m_dollars(stock_dormant)} (produits sans relevé de stock : {int(valeurs.isna().sum())})")
tresorerie = pd.DataFrame({"part du stock dormant récupérée": [0.25, 0.50, 0.75]})
tresorerie["trésorerie libérée (une fois)"] = tresorerie["part du stock dormant récupérée"] * stock_dormant
display(tresorerie.style.format({"part du stock dormant récupérée": "{:.0%}", "trésorerie libérée (une fois)": "{:,.0f}"}).hide(axis="index"))
tableaux["8_stock_dormant"] = tresorerie

In [ ]:
# ---------------------------------------------------------------------------
# Le tableau des leviers
# ---------------------------------------------------------------------------
leviers = pd.DataFrame(leviers).set_index("levier").sort_values("gain par an", ascending=False)
print(f"Résultat d'exploitation de l'exercice {EXERCICE} : {m_dollars(resultat_an)} · magasins : {m_dollars(resultat_magasins_an)}")
print(f"Les leviers de rentabilité, chiffrés sur l'exercice {EXERCICE}, classés par gain :")
display(leviers.style.format({"gain par an": "{:,.0f}", "résultat d'exploitation après": "{:,.0f}", "résultat des magasins après": "{:,.0f}",
                              "hausse du résultat": "{:+.1%}", "baisse des ventes qui annule le gain": "{:.1%}"}, na_rep="sans objet"))
tableaux["8_leviers"] = leviers

# Par famille, sans double compte : charges au montant de la référence, publicité à −10 %
sans_double = leviers[~leviers.index.str.contains(f"montant de {BASCULE}|Regrouper|de 5 %|de 20 %|^dont")]
par_famille = sans_double.groupby("famille")["gain par an"].sum().sort_values(ascending=False)
print("Par famille de leviers, sans double compte :")
display(par_famille.to_frame().style.format("{:,.0f}"))
tableaux["8_leviers_par_famille"] = par_famille.to_frame()

# ---------------------------------------------------------------------------
# Grille : que devient le résultat des magasins si les ventes réagissent à une baisse de publicité ?
# ---------------------------------------------------------------------------
reactions = [0, -0.02, -0.05, -0.08, -0.10]
grille_pub = pd.DataFrame({f"publicité {-b:+.0%}".replace("%", " %"): [resultat_magasins_an + b * publicite_magasins_an + r * ca_magasins_an * TAUX_CONTRIBUTION_MAGASINS
                                                                       for r in reactions] for b in [0.05, 0.10, 0.20]},
                          index=[f"ventes {r:+.0%}".replace("%", " %") if r else "ventes inchangées" for r in reactions])
print(f"Résultat d'exploitation des magasins en {EXERCICE} ({m_dollars(resultat_magasins_an)}) selon la baisse de publicité et la réaction des ventes :")
display(grille_pub.style.format("{:,.0f}").map(lambda v: f"color: {viz.ALERTE}" if v < 0 else ""))
tableaux["8_grille_publicite"] = grille_pub

# ---------------------------------------------------------------------------
# Graphique des leviers
# ---------------------------------------------------------------------------
COULEURS_FAMILLE = {"Assortiment": viz.PRINCIPALE, "Charges": viz.CATEGORIES[2], "Publicité": viz.ALERTE,
                    "Promotions": viz.ACCENT, "Retours": viz.GRIS, "Ventes": viz.CATEGORIES[4]}
trace = leviers.sort_values("gain par an")
fig, ax = plt.subplots(figsize=(10.5, 6.4))
libelles = [textwrap.fill(nom, 50) for nom in trace.index]
ax.barh(libelles, trace["gain par an"], color=[COULEURS_FAMILLE[f] for f in trace.famille])
for y, v in enumerate(trace["gain par an"]):
    ax.text(v, y, "  " + m_dollars(v), va="center", fontsize=8.5, color=viz.FONCE)
for famille, couleur in COULEURS_FAMILLE.items():
    if famille in set(trace.famille):
        ax.barh([libelles[0]], [0], color=couleur, label=famille)
ax.legend(frameon=False, loc="lower right")
viz.format_dollars(ax, axe="x")
ax.set_xlim(0, trace["gain par an"].max() * 1.18)
ax.tick_params(axis="y", labelsize=8.5)
viz.titre(ax, "Renouveler l'assortiment et revenir sur les charges : les premiers leviers",
          f"Gain de résultat d'exploitation par an, chiffré sur l'exercice {EXERCICE}")
print("Figure enregistrée :", viz.enregistrer(fig, "06_leviers_de_rentabilite"))
plt.show()

**Lecture**

**Les leviers, du plus rentable au moins rentable, sur l'exercice 2021.**

| Famille | Levier | Gain par an | Hausse du résultat | Résultat des magasins après | Baisse des ventes qui annule le gain |
|---|---|---|---|---|---|
| Assortiment | Revenir au rythme de lancement de 2019 (552 nouveautés au lieu de 272) | 46,1 M$ | +32,5 % | | |
| Publicité | Baisser la publicité des magasins de 20 % | 12,4 M$ | +8,7 % | +1,6 M$ | 8,5 % |
| Charges | Ramener personnel, administration et informatique au montant de 2019 | 9,2 M$ | +6,5 % | −1,5 M$ | |
| Charges | Regrouper administration et informatique par pays (30 % du poste) | 6,6 M$ | +4,7 % | −4,2 M$ | |
| Publicité | Baisser la publicité des magasins de 10 % | 6,2 M$ | +4,4 % | −4,6 M$ | 4,3 % |
| Promotions | Supprimer les remises en magasin | 5,4 M$ | +3,8 % | −5,4 M$ | 5,2 % des ventes en période de promotion (`07`, `09`) |
| Charges | Ramener les mêmes comptes au montant de 2020 | 4,8 M$ | +3,4 % | −6,0 M$ | |
| Promotions | dont : suspendre la promotion à 20 % en magasin (octobre à janvier) | 3,9 M$ | +2,7 % | −6,9 M$ | 9,2 % des ventes de la période (`07`) |
| Assortiment | Vendre les nouveautés en magasin aussi bien que les produits anciens | 3,8 M$ | +2,7 % | −7,0 M$ | |
| Publicité | Baisser la publicité des magasins de 5 % | 3,1 M$ | +2,2 % | −7,7 M$ | 2,1 % |
| Promotions | Plafonner la remise à 10 % en ligne et chez le revendeur (à tester) | 2,9 M$ | +2,1 % | | |
| Retours | Réduire les retours de moitié | 2,6 M$ | +1,8 % | | |
| Ventes | Déplacer 1 % du CA de la gamme économique vers le luxe | 1,1 M$ | +0,8 % | | |

**Comment lire ce tableau.**

- **Renouveler l'assortiment est de loin le premier levier** : 280 nouveautés de plus par an, à 298 114 $ de chiffre d'affaires chacune comme celles de 2021, rapporteraient 46,1 M$. C'est un **gain brut** : le coût de conception et de lancement d'une nouveauté n'est pas dans nos données, il faut le demander aux achats. Le gain va surtout à la boutique en ligne et au revendeur, qui vendent les deux tiers des nouveautés.
- **Remettre les nouveautés en magasin** (3,8 M$) : en 2021, un magasin vend une nouveauté à 0,80 fois ce qu'en vend la boutique en ligne, contre 1,00 pour un produit ancien. Mettre en rayon et en avant les nouveautés dans tous les magasins ramènerait leurs ventes en magasin de 27,5 à 34,4 M$.
- **Revenir sur les hausses de frais généraux** (9,2 M$, ou 4,8 M$ en annulant seulement celle de 2021) est un levier **sans risque sur les ventes** : ces charges n'ont aucun lien avec ce que vend un magasin (section 5). Les ramener à leur part des ventes de 2019 rapporterait 32,1 M$, mais demanderait de les couper de 60 % : hors de portée en un exercice.
- **Le regroupement des bureaux par pays** est une piste de réflexion : 4,4 à 8,8 M$ selon que 20 % ou 40 % du poste disparaît, sur 22,1 M$ d'administration et d'informatique dans les cinq pays à plusieurs magasins. La part réellement économisable relève des gens du métier. Il recoupe le retour au montant de 2019 : **ne pas additionner les deux**.
- **Baisser la publicité** est le levier le plus rapide, et le plus incertain. Aucun test ne montre que les ventes suivent la publicité, aucun ne l'exclut. À −20 %, les magasins reviennent juste à l'équilibre (+1,6 M$) si les ventes ne bougent pas, mais repassent en déficit (−1,3 M$) si elles reculent de 2 %. Le gain disparaît au-delà de 8,5 % de ventes perdues.
- **Supprimer les remises en magasin** (5,4 M$) suppose que les ventes remisées auraient eu lieu sans remise ; `09_q5_remises` montre qu'en magasin une ligne remisée ne contient pas plus d'unités qu'une autre, sur les trois exercices. La promotion à 20 % (octobre à janvier) en fait 3,9 M$ : c'est la première à suspendre, et une **partie** de ce levier, pas un levier à additionner.
- **Plafonner la remise à 10 % en ligne et chez le revendeur** (2,9 M$) est une piste à tester avec un groupe témoin : dans ces deux canaux, une ligne remisée est plus grosse, sans qu'on sache si la remise en est la cause (`09_q5_remises`).
- **Remises et retours sont calculés nets des autres charges** : une remise supprimée ou un retour évité laisse la vente dans le chiffre d'affaires, et les autres charges (environ 1 cent par dollar de vente) s'y appliquent. La remise de 5,48 M$ rapporte donc 5,42 M$, la moitié des retours (marge perdue de 5,34 M$) 2,62 M$.
- Les retours (2,6 M$) et la gamme (1,1 M$) sont des leviers mineurs, et le second va contre la descente en gamme observée.

**Par famille, sans double compte** (charges au montant de 2019, publicité à −10 %, sans le regroupement) : assortiment 49,8 M$, charges 9,2 M$, promotions 8,4 M$ (magasins 5,4 M$, plafond en ligne et chez le revendeur 2,9 M$), publicité 6,2 M$, retours 2,6 M$, ventes 1,1 M$, soit **77,3 M$**. Ce total est un ordre de grandeur : les leviers ne s'additionnent pas exactement (une nouveauté de plus en magasin rend aussi une remise de moins nécessaire, par exemple).

**Ce qui rend les magasins bénéficiaires sans toucher aux ventes** : ramener personnel, administration et informatique au montant de 2019 et baisser la publicité de 10 % donnent ensemble +4,6 M$ de résultat aux magasins (−10,8 + 9,2 + 6,2).

**Le stock dormant est de la trésorerie, pas du résultat.** 74 453 couples magasin × produit sur 126 352 (59 %) n'ont enregistré aucune vente en 2021. Leur stock est estimé à 180,2 M$ au coût. En récupérer 25 %, 50 % ou 75 % (transfert vers la boutique en ligne ou le revendeur, déstockage) libérerait 45, 90 ou 135 M$, **une seule fois**. L'estimation repose sur des relevés de stock partiels et sur l'indice « stock dormant » de `06b` : à confirmer par un inventaire.

## 9. Synthèse : réponse à la question Q1

**Question Q1** : *quels sont les facteurs déterminants de la performance des ventes (boutique, produit, promotion) ?*

### 1. Ce qui fait la performance des ventes

| Niveau | Facteur déterminant | Mesure | Facteurs sans effet |
|---|---|---|---|
| Une vente | Le produit et son prix | η² 86,1 % (produit), 77,6 % (prix) | boutique 0,6 %, pays 0,3 %, jour et jour férié 0,0 % |
| Un magasin, un mois | Le nombre de ventes | 76,2 % du chiffre d'affaires | surface, effectif, sévérité locale du COVID-19 |
| D'un exercice à l'autre | Ce qui change pour toute l'entreprise : l'assortiment | η² de l'année 69,7 % (79,0 % sans Moscow No.2) | saison (profil stable), pays |

- **Le produit** est le premier facteur : la catégorie, et surtout la gamme, fixent la marge (48,9 % en économique, 65,5 % en luxe).
- **La boutique** n'est pas un facteur de performance : les magasins se ressemblent (la moitié se tient à moins de 2 % de la médiane), sauf Moscow No.2. La différence de performance est entre **canaux** : magasins −48,6 %, en ligne +6,7 %, revendeur −14,1 % en deux exercices.
- **La promotion** n'est pas un facteur de performance : aucun effet sur les ventes n'est démontré (`07_q2_promotions`), et la corrélation de −0,44 avec le résultat d'un magasin disparaît à mois égal (−0,02).

### 2. Ce qui a fait baisser les ventes

Le chiffre d'affaires passe de 945,4 M$ à 759,1 M$ puis 688,2 M$ (−27,2 %). 2020 est une baisse de volume, 2021 une baisse de prix. `06b` en établit la cause : **un assortiment qui vieillit faute de nouveautés** (552, 429 puis 272 lancements), des produits vendus dans deux fois moins de magasins, une clientèle qui descend en gamme dans tous les canaux, et des grands comptes en ligne qui partent quand les nouveautés s'épuisent.

### 3. Ce qui a fait basculer la rentabilité

| | 2019 | 2020 | 2021 |
|---|---|---|---|
| Résultat d'exploitation de l'entreprise | 290,7 M$ | 184,6 M$ | 141,7 M$ |
| Résultat d'exploitation des magasins | 121,6 M$ | 13,0 M$ | −10,8 M$ |
| Magasins bénéficiaires | 48 sur 48 | 43 sur 51 | 4 sur 53 |
| Frais généraux des magasins ÷ CA | 16,4 % | 29,3 % | 36,8 % |

- **Le marché** : la baisse des ventes coûte 139,2 M$ de marge aux magasins en deux exercices.
- **Les décisions de gestion** : la publicité a été réduite de 24 % (+19,8 M$), mais le personnel, l'administration et l'informatique ont été augmentés dans les 48 magasins (−12,9 M$). Ces hausses dépassent le déficit de 2021.
- **La règle d'imputation** : chaque boutique reçoit le même budget de publicité ; imputé selon les ventes, le même budget laisserait les 53 magasins bénéficiaires.
- Les magasins bénéficiaires de 2021 sont ceux dont les frais généraux pèsent peu (20 % contre 39 %) : parce que leurs ventes ont résisté, ou parce que leurs charges sont fixées sur leur activité actuelle.

### 4. Registre des constats

| Constat | Verdict |
|---|---|
| Le montant d'une vente dépend du produit et de son prix, pas de la boutique ni du jour | Retenu |
| La baisse est un changement de niveau, pas de saison ; aucun effet des jours fériés ni du jour de la semaine | Retenu |
| La pandémie distingue un magasin d'un autre | Non soutenu |
| La surface et l'effectif expliquent les ventes d'un magasin | Non soutenu |
| La perte des magasins vient de frais généraux qui ne suivent pas les ventes | Retenu |
| Les hausses de frais généraux depuis 2019 sont des décisions prises dans tous les magasins | Retenu |
| Les ventes suivent la publicité | Non soutenu (non mesurable sans test piloté) |
| La réduction de la publicité a amélioré le résultat | Retenu |
| Le déficit des magasins tient en partie à la règle de répartition de la publicité | Retenu |
| La publicité des Fêtes est mal calée sur le calendrier des ventes | Ouvert (question au marketing) |
| Les magasins déficitaires sont proches de leur seuil (+12,5 % de ventes) | Retenu |
| Les promotions et les retours expliquent la dégradation du résultat | Non soutenu : leur poids est bien plus faible que celui des charges, de l'assortiment et des grands comptes |

### 5. Les problèmes à régler, par ordre de poids

Les promotions et les retours ne sont pas les problèmes prioritaires : leur poids sur le résultat est bien plus faible que celui de la hausse des frais généraux, du non-renouvellement de l'assortiment et de la perte des grands comptes en ligne.

| Problème | Ce qu'il pèse | Levier sur l'exercice 2021 | Source |
|---|---|---|---|
| Assortiment non renouvelé | Lancements 552 → 429 → 272 ; les nouveautés de l'année portent 69 % de la chute des ventes des magasins | 46,1 M$ (rythme de 2019, gain brut) + 3,8 M$ (nouveautés en rayon) | `06b`, section 8 |
| Perte des grands comptes en ligne | Achats des entreprises clientes historiques : 169,0 → 86,6 → 0,1 M$ | lié au renouvellement de l'assortiment | `06b` (autre table : ne s'additionne pas avec nos montants) |
| Hausse des frais généraux | +26,3 M$ en deux exercices pour l'entreprise (155,0 → 181,3 M$), dont +12,9 M$ en magasin, plus que leur déficit | 9,2 M$ (retour au montant de 2019) | sections 7 et 8 |
| Publicité des magasins | 61,9 M$, 23,5 % de leurs ventes, sans effet mesurable | 6,2 M$ (−10 %) | sections 7 et 8 |
| Remises | 13,9 M$ toutes boutiques, 5,5 M$ en magasin ; celles des magasins ont **baissé** de 6,9 M$ depuis 2019 pendant que leur résultat perdait 132,4 M$ | 5,4 M$ (magasins) + 2,9 M$ (plafond à tester) | `07`, `09` |
| Retours | 5,34 M$ de marge perdue, fréquence stable depuis 2019 | 2,6 M$ (moitié des retours) | `08` |

Remises en magasin et retours réunis valent 8,0 M$ par an : moins d'un cinquième du seul levier sur l'assortiment. Ils restent utiles, mais ils n'expliquent pas la dégradation et ne la régleront pas.

### 6. Recommandation : deux actions complémentaires

1. **Traiter la cause : renouveler l'assortiment.** Revenir au rythme de lancement de 2019 (46,1 M$ de gain brut par an, coût de lancement à chiffrer) et mettre les nouveautés en rayon dans tous les magasins (3,8 M$). Libérer le stock qui ne tourne pas (45 à 135 M$ de trésorerie, une fois).
2. **Traiter la conséquence : recaler les charges sur les ventes.** Annuler les hausses de personnel, d'administration et d'informatique (4,8 M$ dès l'exercice suivant, 9,2 M$ pour revenir à 2019), étudier le regroupement des bureaux par pays, baisser la publicité de 10 % (6,2 M$) en lançant un test piloté avant d'aller plus loin. Ces deux mesures suffisent à rendre les magasins bénéficiaires (+4,6 M$) sans attendre le retour des ventes. Supprimer les remises en magasin (5,4 M$) porterait ce résultat vers 10 M$ (ordre de grandeur), à condition que les ventes tiennent.

Les deux actions ne s'opposent pas : la première rend des ventes aux magasins, la seconde les rend rentables au niveau de ventes actuel.

### 7. Points à valider avec les gens du métier

- le coût de conception et de lancement d'une nouveauté, qui réduit le gain brut du premier levier ;
- la part de l'administration et de l'informatique réellement économisable par un regroupement (20 %, 30 % ou 40 %) ;
- la valeur du stock dormant, estimée à partir de relevés partiels : à confirmer par un inventaire ;
- la date réelle de diffusion de la publicité des Fêtes, engagée de janvier à mars.

## Enregistrement des tableaux

In [ ]:
# Tous les tableaux de ce notebook dans un classeur Excel, un onglet par tableau,
# pour les reprendre dans le rapport d'analyse (livrable L4) sans relancer le code.
chemin_excel = analyse.exporter_tableaux(tableaux, config.L4 / "q1_tableaux.xlsx")
print("Classeur enregistré :", chemin_excel)
print("Onglets :", list(tableaux))